In this notebook
1. Load WQP/main PFAS data
2. Assign state
3. Assign lat and long 
4. Import cleaned Phase I, Phase II and MA files
5. Clean result values
6. Apply PFAS detection rule
7. Standardize columns
8. Combine all PFAS datasets
9. Save final cleaned analysis file

## 1. Load WQP/main PFAS data

In [1]:
import pandas as pd
import numpy as np
import os

In [2]:
file_path = "G:/My Drive/Projects/Large projects/PFAS in water/"

# I load the three WQP PFAS result files.
data_2022 = pd.read_csv(
    file_path + "raw_data/PFAS-2022/resultphyschem (3)/resultphyschem 2022.csv",
    low_memory=False
)

data_2023 = pd.read_csv(
    file_path + "raw_data/PFAS-2023/resultphyschem (3)/resultphyschem 2023.csv",
    low_memory=False
)

data_2024 = pd.read_csv(
    file_path + "raw_data/PFAS-2024/resultphyschem (3)/resultphyschem 2024.csv",
    low_memory=False
)

print("2022:", data_2022.shape)
print("2023:", data_2023.shape)
print("2024:", data_2024.shape)

2022: (58614, 81)
2023: (56771, 81)
2024: (59974, 81)


In [3]:
# I add a year column before combining the WQP files.

data_2022 = data_2022.copy()
data_2023 = data_2023.copy()
data_2024 = data_2024.copy()

data_2022["year"] = 2022
data_2023["year"] = 2023
data_2024["year"] = 2024

In [4]:
# I combine all WQP years into one dataframe.
# I use reindex because the three years may not have exactly the same columns.

all_columns = sorted(
    set(data_2022.columns)
    | set(data_2023.columns)
    | set(data_2024.columns)
)

data_all = pd.concat(
    [
        data_2022.reindex(columns=all_columns),
        data_2023.reindex(columns=all_columns),
        data_2024.reindex(columns=all_columns),
    ],
    ignore_index=True
)

print("Combined WQP rows:", len(data_all))
print(data_all["year"].value_counts(dropna=False).sort_index())

Combined WQP rows: 175359
year
2022    58614
2023    56771
2024    59974
Name: count, dtype: int64


### ActivityMediaSubdivisionName cleaning 

In [5]:
# I keep only water samples.

data_water = data_all.loc[
    data_all["ActivityMediaName"]
    .astype("string")
    .str.strip()
    .eq("Water")
].copy()

print("Rows after Water filter:", len(data_water))
print(data_water["ActivityMediaSubdivisionName"].value_counts(dropna=False))

Rows after Water filter: 102657
ActivityMediaSubdivisionName
Surface Water                          41810
Groundwater                            39771
NaN                                    17995
Drinking Water                           840
Wastewater Treatment Plant Influent      381
Wastewater Treatment Plant Effluent      380
Solids                                   342
Wet Fall Material                        338
Industrial Effluent                      304
Effluent                                 260
Finished Water                           160
Elutriation                               76
Name: count, dtype: int64


In [6]:
# I keep surface water and blank subdivision rows.
# I inspected this in the earlier notebook and many blank subdivision rows are surface water.

mask_surface_or_blank = (
    data_water["ActivityMediaSubdivisionName"].isna()
    |
    data_water["ActivityMediaSubdivisionName"]
    .astype("string")
    .str.strip()
    .eq("Surface Water")
)

data_surface = data_water.loc[mask_surface_or_blank].copy()

print("Rows after Surface Water + blank filter:", len(data_surface))
print(data_surface["ActivityMediaSubdivisionName"].value_counts(dropna=False))

Rows after Surface Water + blank filter: 59805
ActivityMediaSubdivisionName
Surface Water    41810
NaN              17995
Name: count, dtype: int64


In [7]:
display(
    data_surface["ResultAnalyticalMethod/MethodName"]
    .value_counts(dropna=False)
)

ResultAnalyticalMethod/MethodName
PFAS, wu, AXYS MLA-110; EPA 1633                                                                                                                                                 16986
Analysis of PFAS in Aqueous, Solid, Biosolids, and Tissue Samples by LC-MS/MS by EPA 1633 modified                                                                               15240
Analysis of Per- and Polyfluoroalkyl Substances (PFAS) in Aqueous, Solid, Biosolids, and Tissue Samples by LC-MS/MS                                                              10305
Analytical Procedure for the Analysis of PFAS  by LC/MS/MS Isotope Dilution revision 2                                                                                            4440
PFAS SPE/LC/MS/MS SGS FL EPA537M                                                                                                                                                  3249
NaN                                                

In [8]:
# ok now I'm inspection those that used drinking water method, it might be only method and not having any meaning but inspecting them
target_methods = [
    "Determination of Selected Per- and Polyfluorinated Alkyl Substances in Drinking Water by Solid Phase Extraction and Liquid Chromatography/Tandem Mass Spectrometry (LC/MS/MS)"
]

display(
    data_surface.loc[
        data_surface["ResultAnalyticalMethod/MethodName"].isin(target_methods),
        "ActivityMediaSubdivisionName"
    ].value_counts(dropna=False)
)

ActivityMediaSubdivisionName
NaN              232
Surface Water    126
Name: count, dtype: int64

In [9]:
target_methods = [
    "PFAS in Drinking Water by Isotope Dilution Anion Exchange SPE and LC/MS/MS"
]

display(
    data_surface.loc[
        data_surface["ResultAnalyticalMethod/MethodName"].isin(target_methods),
        "ActivityMediaSubdivisionName"
    ].value_counts(dropna=False)
)

ActivityMediaSubdivisionName
NaN              1446
Surface Water     175
Name: count, dtype: int64

In [10]:
# Ok I decided to keep them all

## 2. Assigning lat and long 

In [11]:
import requests
import time
# before assigning state based on lat and long I need to fill missing lat and longs by MonitoringLocationIdentifier
# I make sure latitude and longitude are numeric first.

data_surface["ActivityLocation/LatitudeMeasure"] = pd.to_numeric(
    data_surface["ActivityLocation/LatitudeMeasure"],
    errors="coerce"
)

data_surface["ActivityLocation/LongitudeMeasure"] = pd.to_numeric(
    data_surface["ActivityLocation/LongitudeMeasure"],
    errors="coerce"
)

# I find rows that are missing latitude or longitude.

mask_missing_coords = (
    data_surface["ActivityLocation/LatitudeMeasure"].isna()
    | data_surface["ActivityLocation/LongitudeMeasure"].isna()
)

print("Rows missing coordinates before USGS API:", mask_missing_coords.sum())

display(
    data_surface.loc[
        mask_missing_coords,
        [
            "MonitoringLocationIdentifier",
            "MonitoringLocationName",
            "ActivityLocation/LatitudeMeasure",
            "ActivityLocation/LongitudeMeasure"
        ]
    ].head(50)
)

Rows missing coordinates before USGS API: 25377


,MonitoringLocationIdentifier,MonitoringLocationName,ActivityLocation/LatitudeMeasure,ActivityLocation/LongitudeMeasure
0,USGS-03104500,NaN,NaN,NaN
1,USGS-03104500,NaN,NaN,NaN
2,USGS-03104500,NaN,NaN,NaN
3,USGS-03104500,NaN,NaN,NaN
4,USGS-03104500,NaN,NaN,NaN
5,USGS-03104500,NaN,NaN,NaN
6,USGS-03104500,NaN,NaN,NaN
7,USGS-03104500,NaN,NaN,NaN
8,USGS-03104500,NaN,NaN,NaN
9,USGS-03104500,NaN,NaN,NaN


In [12]:
def get_usgs_site_coordinates(site_id):
    """
    I use the USGS site service to get latitude and longitude
    from a USGS monitoring location identifier.
    """

    url = "https://waterservices.usgs.gov/nwis/site/"

    params = {
        "format": "rdb",
        "sites": site_id,
        "siteOutput": "expanded"
    }

    try:
        response = requests.get(url, params=params, timeout=20)

        if response.status_code != 200:
            return pd.Series([np.nan, np.nan])

        lines = response.text.splitlines()

        data_lines = [
            line for line in lines
            if not line.startswith("#")
        ]

        if len(data_lines) < 3:
            return pd.Series([np.nan, np.nan])

        header = data_lines[0].split("\t")
        values = data_lines[2].split("\t")

        site_data = dict(zip(header, values))

        lat = pd.to_numeric(
            site_data.get("dec_lat_va"),
            errors="coerce"
        )

        lon = pd.to_numeric(
            site_data.get("dec_long_va"),
            errors="coerce"
        )

        return pd.Series([lat, lon])

    except Exception:
        return pd.Series([np.nan, np.nan])

In [13]:
# I only send USGS-like IDs to the USGS API.

mask_missing_coords = (
    data_surface["ActivityLocation/LatitudeMeasure"].isna()
    | data_surface["ActivityLocation/LongitudeMeasure"].isna()
)

mask_usgs_missing_coords = (
    mask_missing_coords
    & data_surface["MonitoringLocationIdentifier"].astype(str).str.startswith("USGS-")
)

print("USGS rows missing coordinates:", mask_usgs_missing_coords.sum())

usgs_ids_missing_coords = (
    data_surface.loc[
        mask_usgs_missing_coords,
        "MonitoringLocationIdentifier"
    ]
    .dropna()
    .astype(str)
    .str.replace("USGS-", "", regex=False)
    .drop_duplicates()
)

print("Unique USGS site IDs to check:", len(usgs_ids_missing_coords))

USGS rows missing coordinates: 25377
Unique USGS site IDs to check: 334


In [14]:
# I call the USGS API once per unique site ID.

usgs_coord_results = []

for i, site_id in enumerate(usgs_ids_missing_coords, start=1):

    lat, lon = get_usgs_site_coordinates(site_id)

    usgs_coord_results.append(
        {
            "usgs_site_id": site_id,
            "api_latitude": lat,
            "api_longitude": lon
        }
    )

    if i % 25 == 0:
        print(f"Checked {i} of {len(usgs_ids_missing_coords)} sites")

    time.sleep(0.1)

usgs_coord_lookup = pd.DataFrame(usgs_coord_results)

display(usgs_coord_lookup.head())
print("Coordinates found:", usgs_coord_lookup["api_latitude"].notna().sum())

Checked 25 of 334 sites
Checked 50 of 334 sites
Checked 75 of 334 sites
Checked 100 of 334 sites
Checked 125 of 334 sites
Checked 150 of 334 sites
Checked 175 of 334 sites
Checked 200 of 334 sites
Checked 225 of 334 sites
Checked 250 of 334 sites
Checked 275 of 334 sites
Checked 300 of 334 sites
Checked 325 of 334 sites


,usgs_site_id,api_latitude,api_longitude
0,03104500,NaN,NaN
1,01465500,NaN,NaN
2,04012500,NaN,NaN
3,04044500,NaN,NaN
4,030109575,NaN,NaN


Coordinates found: 320


In [15]:
# I create a clean USGS site ID column for merging.

data_surface["usgs_site_id_for_api"] = (
    data_surface["MonitoringLocationIdentifier"]
    .astype(str)
    .str.replace("USGS-", "", regex=False)
)

data_surface = data_surface.merge(
    usgs_coord_lookup,
    left_on="usgs_site_id_for_api",
    right_on="usgs_site_id",
    how="left"
)

# I fill only missing latitude and longitude using the API results.

data_surface["ActivityLocation/LatitudeMeasure"] = data_surface[
    "ActivityLocation/LatitudeMeasure"
].fillna(
    data_surface["api_latitude"]
)

data_surface["ActivityLocation/LongitudeMeasure"] = data_surface[
    "ActivityLocation/LongitudeMeasure"
].fillna(
    data_surface["api_longitude"]
)

# I drop helper columns.

data_surface = data_surface.drop(
    columns=[
        "usgs_site_id_for_api",
        "usgs_site_id",
        "api_latitude",
        "api_longitude"
    ],
    errors="ignore"
)

mask_missing_coords_after_api = (
    data_surface["ActivityLocation/LatitudeMeasure"].isna()
    | data_surface["ActivityLocation/LongitudeMeasure"].isna()
)

print("Rows missing coordinates after USGS API:", mask_missing_coords_after_api.sum())

Rows missing coordinates after USGS API: 2804


## 3. Assigning state

### Assigning state based on lat and long 

In [16]:
import geopandas as gpd
from shapely.geometry import Point

In [17]:
# Create point geometry from lat/long
gdf = gpd.GeoDataFrame(
    data_surface,
    geometry=gpd.points_from_xy(
        data_surface["ActivityLocation/LongitudeMeasure"],
        data_surface["ActivityLocation/LatitudeMeasure"]
    ),
    crs="EPSG:4326"
)

In [18]:
# Read US states shapefile
states = gpd.read_file(
    file_path + "raw_data/tl_2025_us_state/tl_2025_us_state.shp"
)

In [19]:
data_surface.columns.tolist()

['ActivityBottomDepthHeightMeasure/MeasureUnitCode',
 'ActivityBottomDepthHeightMeasure/MeasureValue',
 'ActivityCommentText',
 'ActivityConductingOrganizationText',
 'ActivityDepthAltitudeReferencePointText',
 'ActivityDepthHeightMeasure/MeasureUnitCode',
 'ActivityDepthHeightMeasure/MeasureValue',
 'ActivityEndDate',
 'ActivityEndTime/Time',
 'ActivityEndTime/TimeZoneCode',
 'ActivityIdentifier',
 'ActivityLocation/LatitudeMeasure',
 'ActivityLocation/LongitudeMeasure',
 'ActivityMediaName',
 'ActivityMediaSubdivisionName',
 'ActivityRelativeDepthName',
 'ActivityStartDate',
 'ActivityStartTime/Time',
 'ActivityStartTime/TimeZoneCode',
 'ActivityTopDepthHeightMeasure/MeasureUnitCode',
 'ActivityTopDepthHeightMeasure/MeasureValue',
 'ActivityTypeCode',
 'AnalysisStartDate',
 'BinaryObjectFileName',
 'BinaryObjectFileTypeCode',
 'CharacteristicName',
 'DataQuality/BiasValue',
 'DataQuality/ConfidenceIntervalValue',
 'DataQuality/LowerConfidenceLimitValue',
 'DataQuality/PrecisionValue'

In [20]:
# Make sure CRS matches
states = states.to_crs(gdf.crs)

In [21]:
states.columns.tolist()

['REGION',
 'DIVISION',
 'STATEFP',
 'STATENS',
 'GEOID',
 'GEOIDFQ',
 'STUSPS',
 'NAME',
 'LSAD',
 'MTFCC',
 'FUNCSTAT',
 'ALAND',
 'AWATER',
 'INTPTLAT',
 'INTPTLON',
 'geometry']

In [22]:
# Spatial join with states
gdf = gpd.sjoin(
    gdf,
    states[["STUSPS", "NAME", "geometry"]],
    how="left",
    predicate="within"
)

# Create state columns
gdf["state"] = gdf["STUSPS"]
gdf["state_name"] = gdf["NAME"]

In [23]:
gdf["state"].value_counts(dropna=False)

state
MN     15212
PA     13421
NY     10588
NaN     2836
NJ      2741
MT      2269
NM      2221
WI      2197
CO      1590
MO      1009
IA       931
AZ       874
MI       728
DE       720
WV       694
PR       558
CA       390
FL       259
VA       228
MD       109
ND        80
NV        76
IL        31
TX        26
HI        17
Name: count, dtype: int64

In [24]:
gdf.loc[
    gdf["state"].isna(),
    "OrganizationFormalName"
].value_counts(dropna=False)

OrganizationFormalName
USGS Pennsylvania Water Science Center                 2195
USGS Michigan Water Science Center                      192
USGS Minnesota Water Science Center                     158
USGS New Mexico Water Science Center                    104
USGS Wisconsin Water Science Center                      96
Pueblo of Sandia Water Quality Program (New Mexico)      32
USGS Colorado Water Science Center                       31
USGS New Jersey Water Science Center                     26
USGS California Water Science Center                      2
Name: count, dtype: int64

In [25]:
# I create a first state assignment using OrganizationFormalName.
# This is based on the state assignment work from the previous notebook.

org_state_dict = {
    "Pueblo of Sandia Water Quality Program (New Mexico)": "NM",
}

mask = gdf["state"].isna()

gdf.loc[mask, "state"] = (
    gdf.loc[mask, "OrganizationFormalName"]
    .map(org_state_dict)
)

In [26]:
gdf["state"].value_counts(dropna=False)

state
MN     15212
PA     13421
NY     10588
NaN     2804
NJ      2741
MT      2269
NM      2253
WI      2197
CO      1590
MO      1009
IA       931
AZ       874
MI       728
DE       720
WV       694
PR       558
CA       390
FL       259
VA       228
MD       109
ND        80
NV        76
IL        31
TX        26
HI        17
Name: count, dtype: int64

In [27]:
gdf.loc[
    gdf["state"].isna(),
    "OrganizationFormalName"
].value_counts(dropna=False)

OrganizationFormalName
USGS Pennsylvania Water Science Center    2195
USGS Michigan Water Science Center         192
USGS Minnesota Water Science Center        158
USGS New Mexico Water Science Center       104
USGS Wisconsin Water Science Center         96
USGS Colorado Water Science Center          31
USGS New Jersey Water Science Center        26
USGS California Water Science Center         2
Name: count, dtype: int64

In [28]:
data_surface = gdf.copy()

In [29]:
data_surface = gdf.drop(
    columns=[
        "geometry",
        "index_right",
        "STUSPS",
        "NAME",
        "state_name"
    ],
    errors="ignore"
).copy()

In [30]:
data_surface.columns.tolist()

['ActivityBottomDepthHeightMeasure/MeasureUnitCode',
 'ActivityBottomDepthHeightMeasure/MeasureValue',
 'ActivityCommentText',
 'ActivityConductingOrganizationText',
 'ActivityDepthAltitudeReferencePointText',
 'ActivityDepthHeightMeasure/MeasureUnitCode',
 'ActivityDepthHeightMeasure/MeasureValue',
 'ActivityEndDate',
 'ActivityEndTime/Time',
 'ActivityEndTime/TimeZoneCode',
 'ActivityIdentifier',
 'ActivityLocation/LatitudeMeasure',
 'ActivityLocation/LongitudeMeasure',
 'ActivityMediaName',
 'ActivityMediaSubdivisionName',
 'ActivityRelativeDepthName',
 'ActivityStartDate',
 'ActivityStartTime/Time',
 'ActivityStartTime/TimeZoneCode',
 'ActivityTopDepthHeightMeasure/MeasureUnitCode',
 'ActivityTopDepthHeightMeasure/MeasureValue',
 'ActivityTypeCode',
 'AnalysisStartDate',
 'BinaryObjectFileName',
 'BinaryObjectFileTypeCode',
 'CharacteristicName',
 'DataQuality/BiasValue',
 'DataQuality/ConfidenceIntervalValue',
 'DataQuality/LowerConfidenceLimitValue',
 'DataQuality/PrecisionValue'

### 3. Import cleaned Phase I, Phase II and MA files

In [31]:
# I import the cleaned Waterkeeper Phase I, Phase II and Massachusetts files from 0_pfas_2026_separate_data notebook

phase1_long = pd.read_csv(
    file_path + "saved files/waterkeeper_phase1_long.csv",
    low_memory=False
)

phase2_long = pd.read_csv(
    file_path + "saved files/phase2_long.csv",
    low_memory=False
)

ma_long = pd.read_csv(
    file_path + "saved files/ma_water_long.csv",
    low_memory=False
)

print("Phase I:", phase1_long.shape)
print("Phase II:", phase2_long.shape)
print("MA:", ma_long.shape)

print("Phase I columns:")
print(phase1_long.columns.tolist())

print("Phase II columns:")
print(phase2_long.columns.tolist())

print("MA columns:")
print(ma_long.columns.tolist())

Phase I: (12312, 14)
Phase II: (2880, 14)
MA: (4240, 14)
Phase I columns:
['MonitoringLocationIdentifier', 'MonitoringLocationName', 'ActivityStartDate', 'OrganizationFormalName', 'state', 'ActivityLocation/LatitudeMeasure', 'ActivityLocation/LongitudeMeasure', 'CharacteristicName', 'ResultMeasureValue', 'ResultMeasureValue_text', 'DetectionQuantitationLimitMeasure/MeasureValue', 'pfas_detected', 'Upstream or Downstream', 'source']
Phase II columns:
['MonitoringLocationIdentifier', 'MonitoringLocationName', 'ActivityStartDate', 'OrganizationFormalName', 'state', 'ActivityLocation/LatitudeMeasure', 'ActivityLocation/LongitudeMeasure', 'CharacteristicName', 'ResultMeasureValue', 'ResultMeasureValue_text', 'DetectionQuantitationLimitMeasure/MeasureValue', 'pfas_detected', 'Upstream or Downstream', 'source']
MA columns:
['Sample ID', 'Sampling Date', 'Waterkeeper Group', 'State', 'Waterbody', 'Upstream or Downstream', 'Latitude', 'Longitude', 'CharacteristicName', 'raw_result', 'raw_result

In [32]:
# I check the PFAS detection classifications that were already
# assigned to Waterkeeper Phase II in the previous notebook.
# Tony here you can see that I've already assigned pfas detected
display(
    phase2_long["pfas_detected"]
    .value_counts(dropna=False)
)

pfas_detected
N    2264
Y     616
Name: count, dtype: int64

In [33]:
# Flagging

## Maine

In [34]:
## adding Maine

In [35]:
# I import the cleaned Waterkeeper Phase I, Phase II,
# Massachusetts and Maine files from the 0_pfas_2026_separate_data notebook.

phase1_long = pd.read_csv(
    file_path + "saved files/waterkeeper_phase1_long.csv",
    low_memory=False
)

phase2_long = pd.read_csv(
    file_path + "saved files/phase2_long.csv",
    low_memory=False
)

ma_long = pd.read_csv(
    file_path + "saved files/ma_water_long.csv",
    low_memory=False
)

maine_long = pd.read_csv(
    file_path + "saved files/maine_surface_water_long.csv",
    low_memory=False
)

print("Phase I:", phase1_long.shape)
print("Phase II:", phase2_long.shape)
print("MA:", ma_long.shape)
print("Maine:", maine_long.shape)

print("\nPhase I columns:")
print(phase1_long.columns.tolist())

print("\nPhase II columns:")
print(phase2_long.columns.tolist())

print("\nMA columns:")
print(ma_long.columns.tolist())

print("\nMaine columns:")
print(maine_long.columns.tolist())

Phase I: (12312, 14)
Phase II: (2880, 14)
MA: (4240, 14)
Maine: (20254, 68)

Phase I columns:
['MonitoringLocationIdentifier', 'MonitoringLocationName', 'ActivityStartDate', 'OrganizationFormalName', 'state', 'ActivityLocation/LatitudeMeasure', 'ActivityLocation/LongitudeMeasure', 'CharacteristicName', 'ResultMeasureValue', 'ResultMeasureValue_text', 'DetectionQuantitationLimitMeasure/MeasureValue', 'pfas_detected', 'Upstream or Downstream', 'source']

Phase II columns:
['MonitoringLocationIdentifier', 'MonitoringLocationName', 'ActivityStartDate', 'OrganizationFormalName', 'state', 'ActivityLocation/LatitudeMeasure', 'ActivityLocation/LongitudeMeasure', 'CharacteristicName', 'ResultMeasureValue', 'ResultMeasureValue_text', 'DetectionQuantitationLimitMeasure/MeasureValue', 'pfas_detected', 'Upstream or Downstream', 'source']

MA columns:
['Sample ID', 'Sampling Date', 'Waterkeeper Group', 'State', 'Waterbody', 'Upstream or Downstream', 'Latitude', 'Longitude', 'CharacteristicName', 'ra

In [36]:
# I assign Maine's state abbreviation to match the state field
# used throughout the national PFAS dataset.

maine_long["state"] = "ME"

In [37]:
# I add a source column to the WQP data before combining.

data_surface["source"] = "WQP"
data_surface["pfas_detected"]=pd.NA 

In [38]:
# I standardize Massachusetts PFAS water data to match data_surface.

ma_long["MonitoringLocationIdentifier"] = ma_long["Sample ID"]
ma_long["ActivityStartDate"] = ma_long["Sampling Date"]
ma_long["OrganizationFormalName"] = ma_long["Waterkeeper Group"]
ma_long["state"] = ma_long["State"]
ma_long["MonitoringLocationName"] = ma_long["Waterbody"]
ma_long["ActivityLocation/LatitudeMeasure"] = ma_long["Latitude"]
ma_long["ActivityLocation/LongitudeMeasure"] = ma_long["Longitude"]
ma_long["CharacteristicName"] = ma_long["CharacteristicName"]
ma_long["ResultMeasureValue"] = ma_long["raw_result"]
ma_long["ResultMeasureValue_text"] = ma_long["raw_result_text"]
ma_long["pfas_detected"] = ma_long["pfas_detected"]
ma_long["MeasureQualifierCode"] = ma_long["MeasureQualifierCode"]
ma_long["DetectionQuantitationLimitMeasure/MeasureValue"] = ma_long[
    "DetectionQuantitationLimitMeasure/MeasureValue"
]

ma_long["source"] = "Massachusetts"

In [39]:
print(data_surface["source"].value_counts(dropna=False))

source
WQP    59805
Name: count, dtype: int64


In [40]:
# I combine WQP, Massachusetts, Maine, Waterkeeper Phase I,
# and Waterkeeper Phase II.

data_surface = pd.concat(
    [
        data_surface,
        ma_long,
        maine_long,
        phase1_long,
        phase2_long
    ],
    ignore_index=True,
    sort=False
)

print(
    "Rows after merge:",
    len(data_surface)
)

print(
    data_surface["source"]
    .value_counts(dropna=False)
)

Rows after merge: 99491
source
WQP                     59805
Maine DEP               20254
Waterkeeper Phase I     12312
Massachusetts            4240
Waterkeeper Phase II     2880
Name: count, dtype: int64


In [41]:
# I confirm that Maine was added correctly and that its
# state and source fields were preserved.

print(
    "Maine records:",
    data_surface["source"].eq("Maine DEP").sum()
)

print(
    "Maine unique locations:",
    data_surface.loc[
        data_surface["source"].eq("Maine DEP"),
        "MonitoringLocationIdentifier"
    ].nunique()
)

print(
    "Maine state values:"
)

display(
    data_surface.loc[
        data_surface["source"].eq("Maine DEP"),
        "state"
    ].value_counts(dropna=False)
)

Maine records: 20254
Maine unique locations: 443
Maine state values:


state
ME    20254
Name: count, dtype: int64

In [42]:
print(data_surface.shape)
display(data_surface["source"].value_counts(dropna=False))
display(data_surface["pfas_detected"].value_counts(dropna=False))

(99491, 143)


source
WQP                     59805
Maine DEP               20254
Waterkeeper Phase I     12312
Massachusetts            4240
Waterkeeper Phase II     2880
Name: count, dtype: int64

pfas_detected
NaN    80060
U      11104
N       5872
Y       2455
Name: count, dtype: int64

In [43]:
print(data_surface.shape)

(99491, 143)


In [44]:
# I check how many rows are missing latitude or longitude.

mask_missing_coords = (
    data_surface["ActivityLocation/LatitudeMeasure"].isna()
    | data_surface["ActivityLocation/LongitudeMeasure"].isna()
)

print("Rows missing coordinates:", mask_missing_coords.sum())

Rows missing coordinates: 2832


In [45]:
print(
    data_surface.loc[
        mask_missing_coords,
        "MonitoringLocationIdentifier"
    ].nunique()
)

15


In [46]:
display(
    data_surface.loc[
        mask_missing_coords,
        [
            "MonitoringLocationIdentifier",
            "source",
            "MonitoringLocationName",
            "ActivityLocation/LatitudeMeasure",
            "ActivityLocation/LongitudeMeasure"
        ]
    ]
    .drop_duplicates()
    .sort_values(
        [
            "source",
            "MonitoringLocationIdentifier"
        ]
    )
    .reset_index(drop=True)
)

,MonitoringLocationIdentifier,source,MonitoringLocationName,ActivityLocation/LatitudeMeasure,ActivityLocation/LongitudeMeasure
0,MEDEP-148525,Maine DEP,18PD1,NaN,NaN
1,USGS-01464020,WQP,NaN,NaN,NaN
2,USGS-01465500,WQP,NaN,NaN,NaN
3,USGS-01638900,WQP,NaN,NaN,NaN
4,USGS-030109575,WQP,NaN,NaN,NaN
5,USGS-03086000,WQP,NaN,NaN,NaN
6,USGS-03104500,WQP,NaN,NaN,NaN
7,USGS-04012500,WQP,NaN,NaN,NaN
8,USGS-04015330,WQP,NaN,NaN,NaN
9,USGS-04024067,WQP,NaN,NaN,NaN


In [47]:
# Flagging
# MAINE
# I remove the Maine monitoring location that has no usable
# coordinates in either the sample-point or site-level fields.
#
# Because the analysis is location-based, I cannot reliably map
# or spatially analyze this monitoring location.

maine_missing_location = "MEDEP-148525"

mask_drop_maine_missing_coords = (
    data_surface["MonitoringLocationIdentifier"]
    .eq(maine_missing_location)
)

print(
    "Maine records removed for missing coordinates:",
    mask_drop_maine_missing_coords.sum()
)

data_surface = (
    data_surface.loc[
        ~mask_drop_maine_missing_coords
    ]
    .copy()
)

print(
    "Rows remaining:",
    len(data_surface)
)

Maine records removed for missing coordinates: 28
Rows remaining: 99463


In [48]:
# I check which sources have missing monitoring location names.

mask_missing_name = (
    data_surface["MonitoringLocationName"].isna()
    | data_surface["MonitoringLocationName"]
        .astype("string")
        .str.strip()
        .eq("")
)

display(
    data_surface.loc[
        mask_missing_name,
        "source"
    ].value_counts(dropna=False)
)

source
WQP    25377
Name: count, dtype: int64

## 4. Assigining monitoringlocationname

In [49]:
# Now assigning correct monitoringlocationname
# I check missing MonitoringLocationName before using the USGS API.

mask_missing_location_name = (
    data_surface["MonitoringLocationName"].isna()
    | data_surface["MonitoringLocationName"].astype(str).str.strip().eq("")
)

print("Rows missing MonitoringLocationName before USGS API:", mask_missing_location_name.sum())

display(
    data_surface.loc[
        mask_missing_location_name,
        [
            "source",
            "MonitoringLocationIdentifier",
            "MonitoringLocationName",
            "ActivityLocation/LatitudeMeasure",
            "ActivityLocation/LongitudeMeasure"
        ]
    ]
    .drop_duplicates()
    .sort_values(["source", "MonitoringLocationIdentifier"])
)

Rows missing MonitoringLocationName before USGS API: 25377


,source,MonitoringLocationIdentifier,MonitoringLocationName,ActivityLocation/LatitudeMeasure,ActivityLocation/LongitudeMeasure
3916,WQP,USGS-01367625,NaN,41.040278,-74.629722
5505,WQP,USGS-01367770,NaN,41.193889,-74.575278
1546,WQP,USGS-01367800,NaN,41.162778,-74.675278
1344,WQP,USGS-01368000,NaN,41.260095,-74.548493
3395,WQP,USGS-01377000,NaN,40.999167,-73.989167
...,...,...,...,...,...
26961,WQP,USGS-435102076122201,NaN,43.850583,-76.206167
25512,WQP,USGS-440040076171501,NaN,44.011028,-76.287444
29743,WQP,USGS-440152076125301,NaN,44.031056,-76.214806
4012,WQP,USGS-50072975,NaN,18.233142,-65.621378


In [50]:
def get_usgs_site_name(site_id):
    """
    I use the USGS site service to get MonitoringLocationName
    from a USGS monitoring location identifier.
    """

    url = "https://waterservices.usgs.gov/nwis/site/"

    params = {
        "format": "rdb",
        "sites": site_id,
        "siteOutput": "expanded"
    }

    try:
        response = requests.get(url, params=params, timeout=20)

        if response.status_code != 200:
            return np.nan

        lines = response.text.splitlines()

        data_lines = [
            line for line in lines
            if not line.startswith("#")
        ]

        if len(data_lines) < 3:
            return np.nan

        header = data_lines[0].split("\t")
        values = data_lines[2].split("\t")

        site_data = dict(zip(header, values))

        return site_data.get("station_nm", np.nan)

    except Exception:
        return np.nan

In [51]:
# I only send USGS-like IDs with missing names to the USGS API.

mask_usgs_missing_location_name = (
    mask_missing_location_name
    & data_surface["MonitoringLocationIdentifier"]
    .astype(str)
    .str.startswith("USGS-")
)

usgs_ids_missing_names = (
    data_surface.loc[
        mask_usgs_missing_location_name,
        "MonitoringLocationIdentifier"
    ]
    .dropna()
    .astype(str)
    .str.replace("USGS-", "", regex=False)
    .drop_duplicates()
)

print("Unique USGS site IDs with missing names:", len(usgs_ids_missing_names))

Unique USGS site IDs with missing names: 334


In [52]:
# I call the USGS API once per unique USGS site ID.

usgs_name_results = []

for i, site_id in enumerate(usgs_ids_missing_names, start=1):

    site_name = get_usgs_site_name(site_id)

    usgs_name_results.append(
        {
            "usgs_site_id": site_id,
            "api_monitoring_location_name": site_name
        }
    )

    if i % 25 == 0:
        print(f"Checked {i} of {len(usgs_ids_missing_names)} sites")

    time.sleep(0.1)

usgs_name_lookup = pd.DataFrame(usgs_name_results)

display(usgs_name_lookup.head())
print(
    "Names found:",
    usgs_name_lookup["api_monitoring_location_name"].notna().sum()
)

Checked 25 of 334 sites
Checked 50 of 334 sites
Checked 75 of 334 sites
Checked 100 of 334 sites
Checked 125 of 334 sites
Checked 150 of 334 sites
Checked 175 of 334 sites
Checked 200 of 334 sites
Checked 225 of 334 sites
Checked 250 of 334 sites
Checked 275 of 334 sites
Checked 300 of 334 sites
Checked 325 of 334 sites


,usgs_site_id,api_monitoring_location_name
0,03104500,"Shenango River at New Castle, PA"
1,01465500,"Neshaminy Creek near Langhorne, PA"
2,04012500,"POPLAR RIVER AT LUTSEN, MN"
3,04044500,"CARP RIVER AT OLD HWY M-28 NEAR MARQUETTE, MI"
4,030109575,"Tunungwant Creek below outfall at Bradford, PA"


Names found: 334


In [53]:
# I merge the API names back into the full dataset.

data_surface["usgs_site_id_for_api"] = (
    data_surface["MonitoringLocationIdentifier"]
    .astype(str)
    .str.replace("USGS-", "", regex=False)
)

data_surface = data_surface.merge(
    usgs_name_lookup,
    left_on="usgs_site_id_for_api",
    right_on="usgs_site_id",
    how="left"
)

# I fill only missing MonitoringLocationName values.

mask_missing_location_name = (
    data_surface["MonitoringLocationName"].isna()
    | data_surface["MonitoringLocationName"].astype(str).str.strip().eq("")
)

data_surface.loc[
    mask_missing_location_name,
    "MonitoringLocationName"
] = data_surface.loc[
    mask_missing_location_name,
    "api_monitoring_location_name"
]

# I drop helper columns.

data_surface = data_surface.drop(
    columns=[
        "usgs_site_id_for_api",
        "usgs_site_id",
        "api_monitoring_location_name"
    ],
    errors="ignore"
)

# I check missing MonitoringLocationName after using the USGS API.

mask_missing_location_name_after_api = (
    data_surface["MonitoringLocationName"].isna()
    | data_surface["MonitoringLocationName"].astype(str).str.strip().eq("")
)

print(
    "Rows missing MonitoringLocationName after USGS API:",
    mask_missing_location_name_after_api.sum()
)

Rows missing MonitoringLocationName after USGS API: 0


In [54]:
display(
    data_surface.loc[
        mask_missing_location_name_after_api,
        [
            "MonitoringLocationIdentifier",
            "ActivityLocation/LatitudeMeasure",
            "ActivityLocation/LongitudeMeasure",
            "source"
        ]
    ]
    .drop_duplicates()
    .sort_values("MonitoringLocationIdentifier")
    .reset_index(drop=True)
)

,MonitoringLocationIdentifier,ActivityLocation/LatitudeMeasure,ActivityLocation/LongitudeMeasure,source


In [55]:
print(
    "Unique missing MonitoringLocationIdentifier:",
    data_surface.loc[
        data_surface["MonitoringLocationName"].isna(),
        "MonitoringLocationIdentifier"
    ].nunique()
)

Unique missing MonitoringLocationIdentifier: 0


In [56]:
print(data_surface.columns.tolist())

['ActivityBottomDepthHeightMeasure/MeasureUnitCode', 'ActivityBottomDepthHeightMeasure/MeasureValue', 'ActivityCommentText', 'ActivityConductingOrganizationText', 'ActivityDepthAltitudeReferencePointText', 'ActivityDepthHeightMeasure/MeasureUnitCode', 'ActivityDepthHeightMeasure/MeasureValue', 'ActivityEndDate', 'ActivityEndTime/Time', 'ActivityEndTime/TimeZoneCode', 'ActivityIdentifier', 'ActivityLocation/LatitudeMeasure', 'ActivityLocation/LongitudeMeasure', 'ActivityMediaName', 'ActivityMediaSubdivisionName', 'ActivityRelativeDepthName', 'ActivityStartDate', 'ActivityStartTime/Time', 'ActivityStartTime/TimeZoneCode', 'ActivityTopDepthHeightMeasure/MeasureUnitCode', 'ActivityTopDepthHeightMeasure/MeasureValue', 'ActivityTypeCode', 'AnalysisStartDate', 'BinaryObjectFileName', 'BinaryObjectFileTypeCode', 'CharacteristicName', 'DataQuality/BiasValue', 'DataQuality/ConfidenceIntervalValue', 'DataQuality/LowerConfidenceLimitValue', 'DataQuality/PrecisionValue', 'DataQuality/UpperConfidenc

In [57]:
# ok final cleaning 
# I 've seen 1 and -1, fixing them
mask_one_one = (
    data_surface["ActivityLocation/LatitudeMeasure"].eq(1)
    & data_surface["ActivityLocation/LongitudeMeasure"].eq(-1)
)

print("Rows with coordinates (1, -1):", mask_one_one.sum())

display(
    data_surface.loc[
        mask_one_one,
        [
            "MonitoringLocationIdentifier",
            "source",
            "MonitoringLocationName",
            "ActivityLocation/LatitudeMeasure",
            "ActivityLocation/LongitudeMeasure"
        ]
    ]
    .drop_duplicates()
)

Rows with coordinates (1, -1): 32


,MonitoringLocationIdentifier,source,MonitoringLocationName,ActivityLocation/LatitudeMeasure,ActivityLocation/LongitudeMeasure
32527,SANDIAWQ_WQX-EB-1,WQP,EB1,1.0,-1.0
47273,SANDIAWQ_WQX-ALBMAIN05,WQP,Albuquerque Main Canal at Pueblo Northern Boun...,1.0,-1.0


In [58]:
#  fix two New Mexico rows that have placeholder coordinates.

nm_coordinate_fixes = {
    "SANDIAWQ_WQX-ALBMAIN05": {
        "lat": 35.159,
        "lon": -106.640
    },
    "SANDIAWQ_WQX-EB-1": {
        "lat": 35.22,
        "lon": -106.60
    }
}

for location_id, coords in nm_coordinate_fixes.items():

    mask = (
        data_surface["MonitoringLocationIdentifier"].eq(location_id)
    )

    data_surface.loc[
        mask,
        "ActivityLocation/LatitudeMeasure"
    ] = coords["lat"]

    data_surface.loc[
        mask,
        "ActivityLocation/LongitudeMeasure"
    ] = coords["lon"]

    print(location_id, "rows fixed:", mask.sum())

SANDIAWQ_WQX-ALBMAIN05 rows fixed: 16
SANDIAWQ_WQX-EB-1 rows fixed: 16


In [59]:
# I save the cleaned surface water dataset before removing QA/QC samples.

output_file = os.path.join(
    file_path,
    "data_surface_before_qaqc_review.csv"
)

data_surface.to_csv(
    output_file,
    index=False
)

print("Saved:", output_file)
print("Rows:", len(data_surface))

Saved: G:/My Drive/Projects/Large projects/PFAS in water/data_surface_before_qaqc_review.csv
Rows: 99463


### Keeping only PFAS records

In [60]:
# I review all analyte names in the WQP surface-water data
# to make sure the downloaded files contain only PFAS compounds.

characteristic_summary = (
    data_surface["CharacteristicName"]
    .value_counts(dropna=False)
    .reset_index()
)

characteristic_summary.columns = [
    "CharacteristicName",
    "records"
]

print(
    "Unique analytes:",
    data_surface["CharacteristicName"].nunique(dropna=False)
)

display(characteristic_summary)

Unique analytes: 264


,CharacteristicName,records
0,Perfluorooctanesulfonamide,1596
1,Fluorotelomer sulfonate 4:2,1406
2,Fluorotelomer sulfonate 8:2,1381
3,Perfluoroundecanoate,1349
4,Perfluoro(4-methoxybutanoic) acid,1260
...,...,...
259,N-Ethyl-perfluorooctane sulfonamido acetic acid,7
260,"Cyclohexanesulfonic acid, 1,2,2,3,3,4,5,5,6,6-...",7
261,Perfluoro-1-octanesulfonamide***retired***use ...,7
262,"1-Propanesulfonic acid, 1,1,2,2,3,3,3-heptaflu...",7


In [61]:
# I display every unique analyte name so I can identify
# any records that are not PFAS compounds.

all_characteristics = (
    data_surface["CharacteristicName"]
    .dropna()
    .astype("string")
    .str.strip()
    .drop_duplicates()
    .sort_values()
    .reset_index(drop=True)
)

pd.set_option("display.max_rows", None)

display(all_characteristics)

0                                      (-)-delta9-THC-d3
1      1-Butanesulfonic acid, 1,1,2,2,3,3,4,4,4-nonaf...
2      1-Decanesulfonic acid, 1,1,2,2,3,3,4,4,5,5,6,6...
3      1-Heptanesulfonic acid, 1,1,2,2,3,3,4,4,5,5,6,...
4      1-Heptanesulfonic acid, 1,1,2,2,3,3,4,4,5,5,6,...
5      1-Hexanesulfonic acid, 1,1,2,2,3,3,4,4,5,5,6,6...
6      1-Hexanesulfonic acid, 3,3,4,4,5,5,6,6,6-nonaf...
7      1-Octanesulfonamide, 1,1,2,2,3,3,4,4,5,5,6,6,7...
8      1-Octanesulfonic acid, 1,1,2,2,3,3,4,4,5,5,6,6...
9      1-Octanesulfonic acid, 3,3,4,4,5,5,6,6,7,7,8,8...
10     1-Octanesulfonic acid, 3,3,4,4,5,5,6,6,7,7,8,8...
11     1-Propanesulfonic acid, 1,1,2,2,3,3,3-heptaflu...
12                                              10:2 FTS
13     11-CHLOROEICOSAFLUORO-3-OXAUNDECANE-1-SULFONIC...
14       11-Chloroeicosafluoro-3-oxaundecane-1-sulfonate
15       11-chloroeicosafluoro-3-oxaundecane-1-sulfonate
16     11-chloroeicosafluoro-3-oxaundecane-1-sulfonic...
17                             

In [62]:
# I remove a non-PFAS analytes
non_pfas_analytes = [
    "(-)-delta9-THC-d3"
]

mask_non_pfas = (
    data_surface["CharacteristicName"]
    .astype("string")
    .str.strip()
    .isin(non_pfas_analytes)
)

print(
    "Non-PFAS records being removed:",
    mask_non_pfas.sum()
)

display(
    data_surface.loc[
        mask_non_pfas,
        [
            "CharacteristicName",
            "MonitoringLocationIdentifier",
            "MonitoringLocationName",
            "OrganizationFormalName",
            "ActivityStartDate",
            "ResultMeasureValue",
            "ResultMeasure/MeasureUnitCode"
        ]
    ]
)

data_surface = (
    data_surface.loc[~mask_non_pfas]
    .copy()
)

print(
    "Remaining records:",
    len(data_surface)
)

Non-PFAS records being removed: 8


,CharacteristicName,MonitoringLocationIdentifier,MonitoringLocationName,OrganizationFormalName,ActivityStartDate,ResultMeasureValue,ResultMeasure/MeasureUnitCode
41649,(-)-delta9-THC-d3,SOUTHUTE-AR 15-4,Historically ANIMAS 1 & SUTANIMS1,Southern Ute Tribe,2024-04-16,1070,ng/L
43052,(-)-delta9-THC-d3,SOUTHUTE-AR 2-7,Animas River 2-7,Southern Ute Tribe,2024-04-16,1900,ng/L
43333,(-)-delta9-THC-d3,SOUTHUTE-FLR 1-2,Florida River 1-2,Southern Ute Tribe,2024-04-16,1330,ng/L
45612,(-)-delta9-THC-d3,SOUTHUTE-AR 10-8,Historically NAR3A,Southern Ute Tribe,2024-04-16,1350,ng/L
58163,(-)-delta9-THC-d3,SOUTHUTE-AR 2-7,Animas River 2-7,Southern Ute Tribe,2024-10-28,817,ng/L
58902,(-)-delta9-THC-d3,SOUTHUTE-AR 15-4,Historically ANIMAS 1 & SUTANIMS1,Southern Ute Tribe,2024-10-28,923,ng/L
59036,(-)-delta9-THC-d3,SOUTHUTE-AR 10-8,Historically NAR3A,Southern Ute Tribe,2024-10-28,1040,ng/L
59454,(-)-delta9-THC-d3,SOUTHUTE-FLR 1-2,Florida River 1-2,Southern Ute Tribe,2024-10-28,933,ng/L


Remaining records: 99455


### 3. Inpsecting QA/QC samples

In [63]:
# I review activity types to identify QA/QC samples.

display(
    data_surface["ActivityTypeCode"]
    .value_counts(dropna=False)
)

ActivityTypeCode
Sample-Routine                                           51228
NaN                                                      39658
Quality Control Sample-Field Replicate                    4166
Quality Control Sample-Field Blank                        2385
Quality Control Sample-Lab Duplicate                       640
Quality Control Field Sample Equipment Rinsate Blank       480
Quality Control Sample-Lab Surrogate Control Standard      288
Quality Control Sample-Blind Duplicate                     280
Quality Control Sample-Trip Blank                          129
Quality Control Sample-Equipment Blank                     115
Sample-Integrated Vertical Profile                          86
Name: count, dtype: int64

In [64]:
# I separate routine and field replicate samples.

routine = data_surface.loc[
    data_surface["ActivityTypeCode"] == "Sample-Routine"
].copy()

replicate = data_surface.loc[
    data_surface["ActivityTypeCode"] == "Quality Control Sample-Field Replicate"
].copy()

# I look for replicate samples that match a routine sample
# at the same location, date, and PFAS parameter.

replicate_matches = replicate.merge(
    routine[
        [
            "MonitoringLocationIdentifier",
            "ActivityStartDate",
            "CharacteristicName",
            "ActivityLocation/LatitudeMeasure",
            "ActivityLocation/LongitudeMeasure",
            "ActivityStartTime/Time"
           
        ]
    ].drop_duplicates(),
    on=[
        "MonitoringLocationIdentifier",
        "ActivityStartDate",
        "CharacteristicName"
    ],
    how="inner"
)

print("Replicate rows:", len(replicate))
print("Replicates matching a routine sample:", len(replicate_matches))

Replicate rows: 4166
Replicates matching a routine sample: 2916


In [65]:
# I count the number of PFAS analytes measured for each event.

routine_events = (
    routine.groupby(
        [
            "MonitoringLocationIdentifier",
            "ActivityStartDate"
        ]
    )["CharacteristicName"]
    .nunique()
    .reset_index(name="routine_parameters")
)

replicate_events = (
    replicate.groupby(
        [
            "MonitoringLocationIdentifier",
            "ActivityStartDate"
        ]
    )["CharacteristicName"]
    .nunique()
    .reset_index(name="replicate_parameters")
)

event_compare = replicate_events.merge(
    routine_events,
    on=[
        "MonitoringLocationIdentifier",
        "ActivityStartDate"
    ],
    how="left"
)

display(event_compare.head())

,MonitoringLocationIdentifier,ActivityStartDate,replicate_parameters,routine_parameters
0,CSKTRIBE-LBR-BLW-HTSP-WWTP,2022-06-28,32,32.0
1,CSKTRIBE-LBR-BLW-HTSP-WWTP,2022-08-30,31,31.0
2,CSKTRIBE-LBR-BLW-HTSP-WWTP,2023-06-08,31,31.0
3,CSKTRIBE-LBR-BLW-HTSP-WWTP,2023-08-01,31,31.0
4,DRBC-BW,2023-08-23,40,40.0


In [66]:
# I identify replicate events that have a matching routine sample
# at the same monitoring location and sampling date.

compare_sets = event_compare.copy()

compare_sets["has_routine_match"] = (
    compare_sets["routine_parameters"].notna()
)

replicate_events_to_drop = compare_sets.loc[
    compare_sets["has_routine_match"]
].copy()

# I create keys so I can match these replicate events back to the full dataset.

replicate_events_to_drop["drop_key"] = (
    replicate_events_to_drop["MonitoringLocationIdentifier"].astype(str)
    + "|"
    + replicate_events_to_drop["ActivityStartDate"].astype(str)
)

data_surface["drop_key"] = (
    data_surface["MonitoringLocationIdentifier"].astype(str)
    + "|"
    + data_surface["ActivityStartDate"].astype(str)
)

replicate["drop_key"] = (
    replicate["MonitoringLocationIdentifier"].astype(str)
    + "|"
    + replicate["ActivityStartDate"].astype(str)
)

replicate_rows_to_drop = replicate.loc[
    replicate["drop_key"].isin(
        replicate_events_to_drop["drop_key"]
    )
].copy()

rows_before = len(data_surface)

data_surface = data_surface.loc[
    ~data_surface.index.isin(replicate_rows_to_drop.index)
].copy()

data_surface = data_surface.drop(columns="drop_key")

print("Replicate events to drop:", len(replicate_events_to_drop))
print("Replicate rows to drop:", len(replicate_rows_to_drop))
print("Rows before:", rows_before)
print("Rows after:", len(data_surface))
print("Rows removed:", rows_before - len(data_surface))

Replicate events to drop: 76
Replicate rows to drop: 2920
Rows before: 99455
Rows after: 96535
Rows removed: 2920


In [67]:
# I identify replicate rows that are dropped at the event level
# even though they do not have an exact matching routine analyte.

routine_analyte_keys = (
    routine[
        [
            "MonitoringLocationIdentifier",
            "ActivityStartDate",
            "CharacteristicName"
        ]
    ]
    .drop_duplicates()
)

replicate_rows_to_drop_check = replicate_rows_to_drop.merge(
    routine_analyte_keys,
    on=[
        "MonitoringLocationIdentifier",
        "ActivityStartDate",
        "CharacteristicName"
    ],
    how="left",
    indicator=True
)

replicate_without_exact_routine_match = (
    replicate_rows_to_drop_check.loc[
        replicate_rows_to_drop_check["_merge"] == "left_only"
    ]
    .copy()
)

print(
    "Dropped replicate rows without an exact routine analyte match:",
    len(replicate_without_exact_routine_match)
)

display(
    replicate_without_exact_routine_match[
        [
            "MonitoringLocationIdentifier",
            "ActivityStartDate",
            "CharacteristicName",
            "ResultMeasureValue",
            "pfas_detected"
        ]
    ]
)

Dropped replicate rows without an exact routine analyte match: 4


,MonitoringLocationIdentifier,ActivityStartDate,CharacteristicName,ResultMeasureValue,pfas_detected
130,USGS-04024067,2022-06-28,"Phenol, 4,4'-[2,2,2-trifluoro-1-(trifluorometh...",NaN,NaN
1304,DRBC-FM,2023-08-23,"Perfluoro-3,6-dioxaheptanoic acid",NaN,NaN
1340,DRBC-FM,2023-08-23,"Perfluoro-3,6-dioxaheptanoic acid",NaN,NaN
1586,DRBC-FM,2023-08-23,"Perfluoro-3,6-dioxaheptanoic acid",NaN,NaN


In [68]:
# I review the routine and field replicate records for the four
# analytes that were dropped without an exact routine match.

check_cases = [
    ("USGS-04024067", "2022-06-28"),
    ("DRBC-FM", "2023-08-23")
]

for location, date in check_cases:

    print("\n", "=" * 80)
    print(location, date)

    check = data_surface.loc[
        (data_surface["MonitoringLocationIdentifier"] == location)
        & (data_surface["ActivityStartDate"].astype(str) == date)
    ].copy()

    display(
        check[
            [
                "MonitoringLocationIdentifier",
                "ActivityStartDate",
                "ActivityTypeCode",
                "CharacteristicName",
                "ResultMeasureValue",
                "ActivityStartTime/Time"
            ]
        ].sort_values(
            ["CharacteristicName", "ActivityTypeCode"]
        )
    )


USGS-04024067 2022-06-28


,MonitoringLocationIdentifier,ActivityStartDate,ActivityTypeCode,CharacteristicName,ResultMeasureValue,ActivityStartTime/Time
11642,USGS-04024067,2022-06-28,Sample-Routine,"1-Octanesulfonic acid, 3,3,4,4,5,5,6,6,7,7,8,8...",NaN,07:35:00
11632,USGS-04024067,2022-06-28,Sample-Routine,11-Chloroeicosafluoro-3-oxaundecane-1-sulfonate,NaN,07:35:00
11637,USGS-04024067,2022-06-28,Sample-Routine,2-(N-methylperfluoro-1-octanesulfonamido)-ethanol,NaN,07:35:00
11631,USGS-04024067,2022-06-28,Sample-Routine,9-Chlorohexadecafluoro-3-oxanonane-1-sulfonate,NaN,07:35:00
11641,USGS-04024067,2022-06-28,Sample-Routine,Fluorotelomer sulfonate 4:2,NaN,07:35:00
11640,USGS-04024067,2022-06-28,Sample-Routine,Fluorotelomer sulfonate 8:2,NaN,07:35:00
11635,USGS-04024067,2022-06-28,Sample-Routine,N-Ethyl-N-(2-hydroxyethyl)perfluorooctanesulfo...,NaN,07:35:00
11629,USGS-04024067,2022-06-28,Sample-Routine,N-Ethylperfluorooctanesulfonamidoacetate,NaN,07:35:00
11628,USGS-04024067,2022-06-28,Sample-Routine,N-Methylperfluorooctanesulfonamidoacetate,NaN,07:35:00
11634,USGS-04024067,2022-06-28,Sample-Routine,N-ethyl Perfluorooctane sulfonamide,NaN,07:35:00



DRBC-FM 2023-08-23


,MonitoringLocationIdentifier,ActivityStartDate,ActivityTypeCode,CharacteristicName,ResultMeasureValue,ActivityStartTime/Time
39045,DRBC-FM,2023-08-23,Sample-Routine,11-chloroeicosafluoro-3-oxaundecane-1-sulfonic...,NaN,12:50:00
36427,DRBC-FM,2023-08-23,Sample-Routine,"2H,2H,3H,3H-Perfluorooctanoate",NaN,12:50:00
36322,DRBC-FM,2023-08-23,Sample-Routine,"3:3 Fluorotelomer carboxylate, ion(1-)",NaN,12:50:00
40220,DRBC-FM,2023-08-23,Sample-Routine,"4,4,5,5,6,6,7,7,8,8,9,9,10,10,10-Pentadecafluo...",NaN,12:50:00
38657,DRBC-FM,2023-08-23,Sample-Routine,"4,8-dioxa-3H-perfluorononanoate",NaN,12:50:00
34099,DRBC-FM,2023-08-23,Sample-Routine,9-Chlorohexadecafluoro-3-oxanonane-1-sulfonic ...,NaN,12:50:00
40132,DRBC-FM,2023-08-23,Sample-Routine,FtS 4:2 ion,NaN,12:50:00
34074,DRBC-FM,2023-08-23,Sample-Routine,FtS 6:2 ion,NaN,12:50:00
37349,DRBC-FM,2023-08-23,Sample-Routine,FtS 8:2 ion,NaN,12:50:00
35449,DRBC-FM,2023-08-23,Sample-Routine,Hexafluoropropylene oxide dimer acid,NaN,12:50:00


In [69]:
## now checking lab duplicates
# I separate lab duplicate samples.

lab_duplicate = data_surface.loc[
    data_surface["ActivityTypeCode"] == "Quality Control Sample-Lab Duplicate"
].copy()

# I look for matching routine samples.

lab_duplicate_matches = lab_duplicate.merge(
    routine[
        [
            "MonitoringLocationIdentifier",
            "ActivityStartDate",
            "CharacteristicName"
        ]
    ].drop_duplicates(),
    on=[
        "MonitoringLocationIdentifier",
        "ActivityStartDate",
        "CharacteristicName"
    ],
    how="inner"
)

print("Lab duplicate rows:", len(lab_duplicate))
print("Rows matching a routine sample:", len(lab_duplicate_matches))

Lab duplicate rows: 640
Rows matching a routine sample: 0


In [70]:
# I separate blind duplicate samples.

blind_duplicate = data_surface.loc[
    data_surface["ActivityTypeCode"] == "Quality Control Sample-Blind Duplicate"
].copy()

# I check whether blind duplicates match routine samples
# at the same location, date, and PFAS parameter.

blind_duplicate_matches = blind_duplicate.merge(
    routine[
        [
            "MonitoringLocationIdentifier",
            "ActivityStartDate",
            "CharacteristicName"
        ]
    ].drop_duplicates(),
    on=[
        "MonitoringLocationIdentifier",
        "ActivityStartDate",
        "CharacteristicName"
    ],
    how="inner"
)

print("Blind duplicate rows:", len(blind_duplicate))
print("Rows matching a routine sample:", len(blind_duplicate_matches))

Blind duplicate rows: 280
Rows matching a routine sample: 280


In [71]:
# I remove blind duplicate QA/QC samples.

data_surface = data_surface.loc[
    data_surface["ActivityTypeCode"]
    != "Quality Control Sample-Blind Duplicate"
].copy()

print("Rows remaining:", len(data_surface))

Rows remaining: 96255


In [72]:
# I review the values reported in ActivityCommentText.

print("Total rows:", len(data_surface))
print(
    "Rows with an ActivityCommentText:",
    data_surface["ActivityCommentText"].notna().sum()
)
print(
    "Rows without an ActivityCommentText:",
    data_surface["ActivityCommentText"].isna().sum()
)
print(
    "Unique non-missing comments:",
    data_surface["ActivityCommentText"].nunique()
)

display(
    data_surface["ActivityCommentText"]
    .value_counts(dropna=False)
    .reset_index(name="count")
)

Total rows: 96255
Rows with an ActivityCommentText: 7277
Rows without an ActivityCommentText: 88978
Unique non-missing comments: 119


,ActivityCommentText,count
0,NaN,88978
1,Field Equipment Blank.,480
2,;,440
3,Yes; Normal Flow | Nothing to note. Nonwadeabl...,228
4,UNFILTERED. Analyzed past the 28 days holding ...,222
5,UNFILTERED. Analyzed past the 28 days holding ...,190
6,UNFILTERED. Analyzed past the 28 days holding ...,190
7,UNFILTERED. Sample received over 6 degree.,190
8,"PFAS added, 2 bottles, group A",185
9,Temperature outside method-required range/No a...,160


In [73]:
# I review records where the activity comment says that PFAS results
# were switched between samples.

mask_switched_results = (
    data_surface["ActivityCommentText"]
    .astype("string")
    .str.contains(
        "PFAS results were switched",
        case=False,
        na=False
    )
)

switched_results = data_surface.loc[
    mask_switched_results
].copy()

print(
    "Records with switched PFAS results comment:",
    len(switched_results)
)

display(
    switched_results[
        [
            "OrganizationFormalName",
            "MonitoringLocationIdentifier",
            "MonitoringLocationName",
            "ActivityStartDate",
            "ActivityTypeCode",
            "CharacteristicName",
            "ResultMeasureValue",
            "ResultMeasure/MeasureUnitCode",
            "MeasureQualifierCode",
            "ResultDetectionConditionText",
            "ActivityCommentText",
            "pfas_detected"
        ]
    ]
)

Records with switched PFAS results comment: 26


,OrganizationFormalName,MonitoringLocationIdentifier,MonitoringLocationName,ActivityStartDate,ActivityTypeCode,CharacteristicName,ResultMeasureValue,ResultMeasure/MeasureUnitCode,MeasureQualifierCode,ResultDetectionConditionText,ActivityCommentText,pfas_detected
11512,USGS New Mexico Water Science Center,USGS-353227105401810,"PECOS RIVER ABOVE GLORIETA CREEK, NM",2022-08-23,Sample-Routine,N-Methylperfluorooctanesulfonamidoacetate,NaN,NaN,NaN,Not Detected,PFAS results were switched with 35350610545561...,NaN
11513,USGS New Mexico Water Science Center,USGS-353227105401810,"PECOS RIVER ABOVE GLORIETA CREEK, NM",2022-08-23,Sample-Routine,N-Ethylperfluorooctanesulfonamidoacetate,NaN,NaN,NaN,Not Detected,PFAS results were switched with 35350610545561...,NaN
11514,USGS New Mexico Water Science Center,USGS-353227105401810,"PECOS RIVER ABOVE GLORIETA CREEK, NM",2022-08-23,Sample-Routine,11-chloroeicosafluoro-3-oxaundecane-1-sulfonate,NaN,NaN,NaN,Not Detected,PFAS results were switched with 35350610545561...,NaN
11515,USGS New Mexico Water Science Center,USGS-353227105401810,"PECOS RIVER ABOVE GLORIETA CREEK, NM",2022-08-23,Sample-Routine,Fluorotelomer sulfonate 4:2,NaN,NaN,NaN,Not Detected,PFAS results were switched with 35350610545561...,NaN
11516,USGS New Mexico Water Science Center,USGS-353227105401810,"PECOS RIVER ABOVE GLORIETA CREEK, NM",2022-08-23,Sample-Routine,"1-Octanesulfonic acid, 3,3,4,4,5,5,6,6,7,7,8,8...",NaN,NaN,NaN,Not Detected,PFAS results were switched with 35350610545561...,NaN
11517,USGS New Mexico Water Science Center,USGS-353227105401810,"PECOS RIVER ABOVE GLORIETA CREEK, NM",2022-08-23,Sample-Routine,Fluorotelomer sulfonate 8:2,NaN,NaN,NaN,Not Detected,PFAS results were switched with 35350610545561...,NaN
11518,USGS New Mexico Water Science Center,USGS-353227105401810,"PECOS RIVER ABOVE GLORIETA CREEK, NM",2022-08-23,Sample-Routine,9-Chlorohexadecafluoro-3-oxanonane-1-sulfonate,NaN,NaN,NaN,Not Detected,PFAS results were switched with 35350610545561...,NaN
11519,USGS New Mexico Water Science Center,USGS-353227105401810,"PECOS RIVER ABOVE GLORIETA CREEK, NM",2022-08-23,Sample-Routine,"4,8-dioxa-3H-perfluorononanoate",NaN,NaN,NaN,Not Detected,PFAS results were switched with 35350610545561...,NaN
11520,USGS New Mexico Water Science Center,USGS-353227105401810,"PECOS RIVER ABOVE GLORIETA CREEK, NM",2022-08-23,Sample-Routine,Perfluoro(2-propoxypropanoate),NaN,NaN,NaN,Not Detected,PFAS results were switched with 35350610545561...,NaN
11521,USGS New Mexico Water Science Center,USGS-353227105401810,"PECOS RIVER ABOVE GLORIETA CREEK, NM",2022-08-23,Sample-Routine,Perfluorobutanoate,NaN,NaN,NaN,Not Detected,PFAS results were switched with 35350610545561...,NaN


### 4. Clean result values

In [74]:
# I review rows with a numeric PFAS result value before assigning
# PFAS detection classifications.

numeric_results = data_surface.loc[
    data_surface["ResultMeasureValue"].notna()
].copy()

print("Rows with numeric ResultMeasureValue_num:", len(numeric_results))

display(
    numeric_results[
        [
           "ResultMeasureValue"
        ]
    ]
    .head(100)
)

Rows with numeric ResultMeasureValue_num: 31705


,ResultMeasureValue
22,6.9
25,3.1
26,7.0
27,1.1
28,10.5
43,0.1
48,2.03
53,0.279
56,0.153
57,0.290


In [75]:
### Before converting all resultmeasurevalue to numeric I'll assign pfas = N to thoese that are ND

In [76]:
# I inspect the remaining unclassified Waterkeeper Phase II records
# to see what information is still available for classification.

phase2_unclassified = data_surface.loc[
    data_surface["pfas_detected"].isna()
    & data_surface["source"].eq("Waterkeeper Phase II")
].copy()

print(
    "Remaining unclassified Waterkeeper Phase II rows:",
    len(phase2_unclassified)
)

display(
    phase2_unclassified[
        [
            "ResultMeasureValue",
            "ResultMeasureValue_text",
            "ResultMeasureValue",
            "ResultDetectionConditionText",
            "MeasureQualifierCode",
            "DetectionQuantitationLimitMeasure/MeasureValue"
        ]
    ].head(20)
)

Remaining unclassified Waterkeeper Phase II rows: 0


,ResultMeasureValue,ResultMeasureValue_text,ResultMeasureValue,ResultDetectionConditionText,MeasureQualifierCode,DetectionQuantitationLimitMeasure/MeasureValue


In [77]:
# I create text and numeric result columns for the WQP data.
# I will use these helper columns throughout the rest of the notebook.

data_surface["ResultMeasureValue_text"] = (
    data_surface["ResultMeasureValue"]
    .astype("string")
    .str.strip()
)

data_surface["ResultMeasureValue_num"] = pd.to_numeric(
    data_surface["ResultMeasureValue_text"],
    errors="coerce"
)

display(
    data_surface[
        ["ResultMeasureValue", "ResultMeasureValue_text", "ResultMeasureValue_num"]
    ].head(20)
)

,ResultMeasureValue,ResultMeasureValue_text,ResultMeasureValue_num
16,NaN,<NA>,<NA>
17,NaN,<NA>,<NA>
18,NaN,<NA>,<NA>
19,NaN,<NA>,<NA>
20,NaN,<NA>,<NA>
21,NaN,<NA>,<NA>
22,6.9,6.9,6.9
23,NaN,<NA>,<NA>
24,NaN,<NA>,<NA>
25,3.1,3.1,3.1


In [78]:
# I count the result values before applying the PFAS detection rule.

print("Non-missing ResultMeasureValue:")
print(data_surface["ResultMeasureValue"].notna().sum())

print("\nTop ResultMeasureValue values:")
display(
    data_surface["ResultMeasureValue_text"]
    .value_counts(dropna=False)
    .head(100)
)

Non-missing ResultMeasureValue:
31705

Top ResultMeasureValue values:


ResultMeasureValue_text
<NA>          64550
< 1 ng/L      10131
<MDL           2128
ND             1100
<2.0            776
< 2 ng/L        505
1.1             255
1.2             228
< 10 ng/L       216
1.3             202
1.8             197
1.6             190
1.5             186
1.9             185
1.7             182
1.4             182
2.2             162
0.8             155
2.1             152
1.0             149
2.3             144
0.6             140
2.6             138
2.5             137
0.7             129
3.1             127
0.5             127
0               127
2.0             117
2.9             113
2.8             112
2.4             112
3.2             107
0.9             104
2.7             102
3.7              96
< 1.4 ng/L       95
3.8              94
3.3              93
3.0              90
4.2              88
3.4              87
3.6              82
3.5              81
4.1              76
2                73
3.9              73
4.4              68
5.2             

### 5. Apply PFAS detection rule

In [79]:
# PFAS detection rule
#
# For this project, we classify PFAS as detected ("Y") whenever there is
# evidence that the analyte was present in the sample
# so:
# I classify PFAS as detected ("Y") when:
# - ResultMeasureValue_num has a positive numeric value, and
#   ResultMeasureValue_text does not begin with "<" or ">".
# - ResultDetectionConditionText indicates the analyte was present.
# - Laboratory comments say the analyte was below the reporting level
#   but at or above the detection level.
# - Positive numeric result values are classified as detected.
# - Laboratory comments such as
#   "below the reporting level but at or above the detection level"
#   are classified as detected because the analyte was found, although the
#   concentration was below the laboratory reporting threshold.

# We classify PFAS as not detected ("N") when:
# - ResultDetectionConditionText indicates "Not Detected".
# - ResultMeasureValue_text is "ND" and no contradiction

# Quality assurance comments such as:
# - "analyte detected in laboratory blank"
# - "holding time exceeded"
# - "sample was diluted"
# - "value verified by rerun"
#
# are not used by themselves to determine detection status because they
# describe data quality issues rather than whether PFAS was present in the
# environmental sample.

In [80]:
# I assign PFAS as detected when the result has a positive numeric value
# and the original result text does not begin with "<" or ">".

result_text_clean = (
    data_surface["ResultMeasureValue_text"]
    .astype("string")
    .str.strip()
)

positive_numeric_result = (
    data_surface["ResultMeasureValue_num"].notna()
    & data_surface["ResultMeasureValue_num"].gt(0)
    & ~result_text_clean.str.startswith("<", na=False)
    & ~result_text_clean.str.startswith(">", na=False)
)

# I preserve PFAS detection classifications that were already assigned
# in the source-specific datasets and only classify remaining records.

mask_new_positive_numeric = (
    data_surface["pfas_detected"].isna()
    & positive_numeric_result
)

print(
    "New rows assigned Y from positive numeric results:",
    mask_new_positive_numeric.sum()
)

data_surface.loc[
    mask_new_positive_numeric,
    "pfas_detected"
] = "Y"

display(
    data_surface["pfas_detected"]
    .value_counts(dropna=False)
)

New rows assigned Y from positive numeric results: 14014


pfas_detected
NaN    62810
Y      16469
U      11104
N       5872
Name: count, dtype: int64

### 1. Define rule based on ResultCommentText column

In [81]:
# Tony said to drop those that are obviously off so saving the file to decide
# I count each non-missing ResultCommentText value.

# I export rows that contain a ResultCommentText for manual review.

comment_review = (
    data_surface.loc[
        data_surface["ResultCommentText"].notna(),
        [
            "MonitoringLocationIdentifier",
            "MonitoringLocationName",
            "ActivityStartDate",
            "CharacteristicName",
            "ResultMeasureValue",
            "ResultDetectionConditionText",
            "pfas_detected",
            "ResultCommentText"
        ]
    ]
    .sort_values(
        [
            "ResultCommentText",
            "CharacteristicName"
        ]
    )
)

comment_review.to_excel(
    "result_comment_review.xlsx",
    index=False
)

print("Rows exported:", len(comment_review))

Rows exported: 22062


In [82]:
# Flagging

In [83]:
# I check whether my general PFAS detection rules classify
# the cleaned Maine DEP records as expected.

mask_maine = (
    data_surface["source"].eq("Maine DEP")
)

print(
    "Maine detection classifications:"
)

display(
    data_surface.loc[
        mask_maine,
        "pfas_detected"
    ].value_counts(dropna=False)
)


# I cross-check Maine's final detection status against whether
# a numeric analytical result was reported.

maine_result_check = pd.crosstab(
    pd.to_numeric(
        data_surface.loc[
            mask_maine,
            "ResultMeasureValue"
        ],
        errors="coerce"
    )
    .notna()
    .map(
        {
            True: "Has numeric result",
            False: "No numeric result"
        }
    ),
    data_surface.loc[
        mask_maine,
        "pfas_detected"
    ],
    margins=True
)

display(maine_result_check)


# I also check the final classification against Maine's
# validation qualifiers.

display(
    pd.crosstab(
        data_surface.loc[
            mask_maine,
            "MeasureQualifierCode"
        ].fillna("MISSING"),
        data_surface.loc[
            mask_maine,
            "pfas_detected"
        ],
        margins=True
    )
)

Maine detection classifications:


pfas_detected
NaN    14718
Y       5508
Name: count, dtype: int64

pfas_detected,Y,All
ResultMeasureValue,,
Has numeric result,5508,5508
All,5508,5508


pfas_detected,Y,All
MeasureQualifierCode,,
E,1,1
J,2971,2971
MISSING,2536,2536
All,5508,5508


In [84]:
# I review laboratory comments that may indicate uncertain PFAS identifications.
#
# I am not excluding these results automatically because several comments
# suggest potential analytical issues rather than confirmed invalid results.
#
# The comments I want to review include:
# - Qualifier peak did not meet method acceptance criteria.
# - Signal-to-noise ratio did not meet method acceptance criteria.
# - Relative retention time did not meet method acceptance criteria.
# - Internal standard recovery issues and matrix effects.
#
# I am especially interested in whether these comments are associated with
# PFAS detections because they could affect how detections are interpreted.
#
# I am not reviewing comments such as U, J, Lab Anote codes, or the PFOS
# lab blank rerun comment because those appear to be qualifiers, laboratory
# notes, or rerun documentation rather than reasons to invalidate results.

suspect_comments = data_surface.loc[
    data_surface["ResultCommentText"]
    .astype(str)
    .str.contains(
        (
            "qualitative identification"
            "|signal to noise"
            "|relative retention time"
            "|matrix effects"
            "|suspect"
        ),
        case=False,
        na=False
    )
].copy()

print("Rows with potentially problematic comments:")
print(len(suspect_comments))

print("\nPFAS detection status:")
display(
    suspect_comments["pfas_detected"]
    .value_counts(dropna=False)
)

print("\nUnique comments:")
display(
    suspect_comments["ResultCommentText"]
    .value_counts(dropna=False)
)

display(
    suspect_comments[
        [
            "MonitoringLocationIdentifier",
            "MonitoringLocationName",
            "ActivityStartDate",
            "CharacteristicName",
            "ResultMeasureValue",
            "ResultDetectionConditionText",
            "pfas_detected",
            "ResultCommentText"
        ]
    ]
    .sort_values(
        [
            "ResultCommentText",
            "CharacteristicName"
        ]
    )
)

Rows with potentially problematic comments:
388

PFAS detection status:


pfas_detected
NaN    386
Y        2
Name: count, dtype: int64


Unique comments:


ResultCommentText
Qualifer peak did not meet the method's acceptance criteria for qualitative identification                                    155
The signal to noise ratio did not meet the method's acceptance criteria for qualitative identification                         92
The extracted internal standards recovered below acceptance criteria due to matrix effects.  Result is considered suspect.     60
The relative retention time did not meet the method's acceptance criteria for qualitative identification                       43
Peak full width half maxima did not meet the method's acceptance criteria for qualitative identification                       15
Peak symmetry did not meet the method's acceptance criteria for qualitative identification                                     14
The qualifier ratio did not meet the method's acceptance criteria for qualitative identification                                6
Peak shape did not meet the method's acceptance criteria for qualitative

,MonitoringLocationIdentifier,MonitoringLocationName,ActivityStartDate,CharacteristicName,ResultMeasureValue,ResultDetectionConditionText,pfas_detected,ResultCommentText
588,USGS-50999987,"CAMP SANTIAGO RAINGAGE SALINAS, PR",2022-06-30,Perfluorobutanesulfonate,NaN,Not Detected,NaN,Peak full width half maxima did not meet the m...
594,USGS-50999987,"CAMP SANTIAGO RAINGAGE SALINAS, PR",2022-06-30,Perfluorobutanoate,NaN,Not Detected,NaN,Peak full width half maxima did not meet the m...
1459,USGS-182723066055200,"STORMWATER STATION AT ARMY AVIATION, SAN JUAN, PR",2022-06-15,Perfluorobutanoate,NaN,Not Detected,NaN,Peak full width half maxima did not meet the m...
4018,USGS-50072975,"STORMWATER STATION AT NSRR CEIBA, PR",2022-06-14,Perfluorobutanoate,NaN,Not Detected,NaN,Peak full width half maxima did not meet the m...
5928,USGS-385822104360601,W FORK ANTLER CR AT MERIDIAN RANCH BLVD NR FALCON,2022-07-07,Perfluorobutanoate,NaN,Not Detected,NaN,Peak full width half maxima did not meet the m...
2165,USGS-01645080,"SENECA CREEK NEAR SENECA, MD",2022-08-04,Perfluorodecanoate,NaN,Not Detected,NaN,Peak full width half maxima did not meet the m...
3868,USGS-01657700,"OCCOQUAN RESERVOIR AT DAM NEAR OCCOQUAN, VA",2022-08-02,Perfluorodecanoate,NaN,Not Detected,NaN,Peak full width half maxima did not meet the m...
4924,USGS-01644600,"GREAT SENECA CREEK NEAR QUINCE ORCHARD, MD",2022-08-04,Perfluorohexanoate,NaN,Not Detected,NaN,Peak full width half maxima did not meet the m...
4926,USGS-01644600,"GREAT SENECA CREEK NEAR QUINCE ORCHARD, MD",2022-08-04,Perfluorononanoate,NaN,Not Detected,NaN,Peak full width half maxima did not meet the m...
5609,USGS-180031066301900,"STORMWATER STATION AT FMS9 BLDG, JUANA DIAZ, PR",2022-06-15,Perfluorooctanesulfonate,NaN,Not Detected,NaN,Peak full width half maxima did not meet the m...


In [85]:
# I remove results with laboratory comments indicating that the analyte
# identification did not meet method acceptance criteria.
#
# These comments include issues such as:
# - Signal-to-noise ratio below acceptance criteria.
# - Relative retention time outside acceptance criteria.
# - Qualifier peak failing qualitative identification criteria.
# - Matrix effects causing suspect results.
#
# All affected rows were classified as not detected, so removing them
# does not remove any PFAS detections from the dataset.

suspect_mask = (
    data_surface["ResultCommentText"]
    .astype(str)
    .str.contains(
        (
            "qualitative identification"
            "|signal to noise"
            "|relative retention time"
            "|matrix effects"
            "|suspect"
        ),
        case=False,
        na=False
    )
)

print("Rows before:", len(data_surface))
print("Rows removed:", suspect_mask.sum())

data_surface = data_surface.loc[
    ~suspect_mask
].copy()

print("Rows after:", len(data_surface))

Rows before: 96255
Rows removed: 388
Rows after: 95867


In [86]:
print(
    "Rows with ResultCommentText:",
    data_surface["ResultCommentText"].notna().sum()
)

Rows with ResultCommentText: 21674


In [87]:
# Rule for ResultCommentText (checked manually)

# I treat PFAS as detected only when ResultCommentText clearly says the analyte
# was detected or found in the sample.
#
# I do not classify rows as detected only because the comment mentions a lab blank,
# blank contamination, non-detect adjustment, or failed qualitative peak criteria.

comment_text = (
    data_surface["ResultCommentText"]
    .astype("string")
    .str.lower()
    .str.strip()
)

mask_result_comment_detected = (
    comment_text.eq("yes_detected")
    | comment_text.str.contains(
        "peak detected but did not meet quantification criteria",
        na=False
    )
)

data_surface.loc[
    data_surface["pfas_detected"].isna()
    & data_surface["ResultMeasureValue_text"].isna()
    & data_surface["ResultMeasureValue_num"].isna()
    & mask_result_comment_detected,
    "pfas_detected"
] = "Y"

print(
    "ResultCommentText detected rule:",
    (
        data_surface["pfas_detected"].eq("Y")
        & mask_result_comment_detected
    ).sum()
)

ResultCommentText detected rule: 146


In [88]:
# I identify results where the analyte was found in both the sample
# and the associated blank.

mask_associated_blank = comment_text.str.contains(
    "analyte found in sample and the associated blank",
    na=False
)

print(
    "Rows with analyte found in sample and associated blank:",
    mask_associated_blank.sum()
)

display(
    data_surface.loc[
        mask_associated_blank,
        [
            "ResultCommentText",
            "ResultMeasureValue_text",
            "ResultMeasureValue_num",
            "MeasureQualifierCode",
            "pfas_detected"
        ]
    ]
)

Rows with analyte found in sample and associated blank: 11


,ResultCommentText,ResultMeasureValue_text,ResultMeasureValue_num,MeasureQualifierCode,pfas_detected
409,analyte found in sample and the associated blank,9.070,9.07,NaN,Y
877,analyte found in sample and the associated blank,12.9,12.9,NaN,Y
1370,analyte found in sample and the associated blank,8.280,8.28,NaN,Y
2217,analyte found in sample and the associated blank,10.1,10.1,NaN,Y
2447,analyte found in sample and the associated blank,2.080,2.08,NaN,Y
2479,analyte found in sample and the associated blank,11.9,11.9,NaN,Y
3659,analyte found in sample and the associated blank,2.570,2.57,NaN,Y
5001,analyte found in sample and the associated blank,8.260,8.26,NaN,Y
5756,analyte found in sample and the associated blank,7.380,7.38,NaN,Y
9247,analyte found in sample and the associated blank,2.490,2.49,NaN,Y


In [89]:
#Tony this is NEW:

# I remove results where the analyte was found in both the sample
# and the associated blank because these results may be affected
# by blank contamination.

mask_associated_blank = (
    data_surface["ResultCommentText"]
    .astype("string")
    .str.lower()
    .str.strip()
    .str.contains(
        "analyte found in sample and the associated blank",
        na=False
    )
)

print(
    "Rows with analyte found in sample and associated blank:",
    mask_associated_blank.sum()
)

data_surface = data_surface.loc[
    ~mask_associated_blank
].copy()

print("Rows remaining after dropping associated-blank results:", len(data_surface))

Rows with analyte found in sample and associated blank: 11
Rows remaining after dropping associated-blank results: 95856


### 2. Define rule based on ResultLaboratoryCommentText column

In [90]:
# I inspect the laboratory comment text first.

# I inspect laboratory comments for records that are still unclassified.

display(
    data_surface.loc[
        data_surface["pfas_detected"].isna(),
        "ResultLaboratoryCommentText"
    ]
    .astype("string")
    .str.strip()
    .value_counts(dropna=False)
)

ResultLaboratoryCommentText
<NA>                                                                                          54208
see result laboratory comment                                                                  3534
IRPIMS VALIDATION                                                                              1297
below the detection levelsee result laboratory comment                                          814
value verified by rerun, same method                                                            445
holding time exceededlab control sample (LCS) recovery outside of range or criteria             285
sample was dilutedsee result laboratory comment                                                 119
[RL=8.86]                                                                                        80
value is highly variable by this methodsee result laboratory comment                             78
[RL=8.56]                                                               

In [91]:
# I create a clean version of ResultLaboratoryCommentText for the
# PFAS detection rules.

lab_comment_text = (
    data_surface["ResultLaboratoryCommentText"]
    .astype("string")
    .str.lower()
    .str.strip()
)

In [92]:
# I review laboratory comments that may indicate contamination
# or analytical interference.
#
# I keep comments related to reporting limits, reruns, dilution,
# holding times, extrapolation, and laboratory control sample recovery
# because these comments do not necessarily invalidate the result.
#
# I exclude comments indicating laboratory blank contamination or
# analytical interference because the laboratory explicitly warns
# that the reported result may not represent a true environmental
# concentration.
#
# The comments targeted for exclusion include:
# - analyte detected in laboratory blank
# - interference from sample matrix
# - result may be affected by interference
# - value reported not confirmable, interference

In [93]:
# I identify results affected by blank contamination or interference.

exclude_mask = (
    data_surface["ResultLaboratoryCommentText"]
    .astype(str)
    .str.contains(
        (
            "analyte detected in laboratory blank"
            "|interference from sample matrix"
            "|result may be affected by interference"
            "|value reported not confirmable, interference"
        ),
        case=False,
        na=False
    )
)

print("Rows flagged for exclusion:", exclude_mask.sum())

display(
    data_surface.loc[
        exclude_mask,
        [
            "CharacteristicName",
            "ResultMeasureValue",
            "ResultDetectionConditionText",
            "pfas_detected",
            "ResultLaboratoryCommentText"
        ]
    ]
)

Rows flagged for exclusion: 95


,CharacteristicName,ResultMeasureValue,ResultDetectionConditionText,pfas_detected,ResultLaboratoryCommentText
120,Perfluorooctanesulfonamide,NaN,Not Detected,NaN,analyte detected in laboratory blank
362,Perfluorobutanesulfonate,NaN,Not Detected,NaN,interference from sample matrixsee result labo...
2011,Perfluorooctanesulfonamide,NaN,Not Detected,NaN,analyte detected in laboratory blank
2047,Perfluorononanoate,0.8,NaN,Y,below the reporting level but at or above the ...
2428,Perfluoroheptanoate,0.7,NaN,Y,below the reporting level but at or above the ...
3296,Perfluorodecanoate,0.9,NaN,Y,below the reporting level but at or above the ...
3468,Perfluorodecanoate,0.8,NaN,Y,below the reporting level but at or above the ...
3516,Perfluorooctanesulfonamide,NaN,Not Detected,NaN,analyte detected in laboratory blank
3980,Perfluorobutanesulfonate,2.0,NaN,Y,result may be affected by interferencebelow th...
4344,Perfluorodecanoate,1.1,NaN,Y,below the reporting level but at or above the ...


In [94]:
display(
    data_surface.loc[
        exclude_mask,
        "ResultLaboratoryCommentText"
    ].value_counts()
)

ResultLaboratoryCommentText
below the reporting level but at or above the detection levelvalue reported not confirmable, interferencesee result laboratory comment    29
analyte detected in laboratory blank                                                                                                      28
below the reporting level but at or above the detection levelanalyte detected in laboratory blank                                         13
value reported not confirmable, interferencesee result laboratory comment                                                                  6
interference from sample matrixsee result laboratory comment                                                                               4
result may be affected by interferencesee result laboratory comment                                                                        3
holding time exceededanalyte detected in laboratory blankbelow the reporting level but at or above the detection level        

In [95]:
## Laboratory comment assumptions

In [96]:
# I remove results where the laboratory explicitly indicates that the
# reported result may be unreliable because of blank contamination or
# analytical interference.
#
# I retain results that were verified by rerun or reported as present
# above the detection level because the laboratory chose to report
# those values despite the qualifier.
#
# This approach removes results that are likely affected by contamination
# or interference while preserving reported PFAS measurements that were
# subsequently verified or considered reportable by the laboratory.

In [97]:
exclude_comments = [
    "analyte detected in laboratory blank",
    "value reported not confirmable, interferencesee result laboratory comment",
    "interference from sample matrixsee result laboratory comment",
    "result may be affected by interferencesee result laboratory comment",
    "result may be affected by interferencebelow the detection levelsee result laboratory comment",
    "result may be affected by interferencevalue extrapolated at low endbelow the detection level",
    "see result laboratory commentsample was dilutedinterference from sample matrix",
    "holding time exceededanalyte detected in laboratory blank",
    "see result laboratory commentanalyte detected in laboratory blank"
]

exclude_mask = (
    data_surface["ResultLaboratoryCommentText"]
    .astype("string")
    .str.strip()
    .isin(exclude_comments)
)

print("Rows before:", len(data_surface))
print("Rows to remove:", exclude_mask.sum())

removed_lab_comment_rows = data_surface.loc[
    exclude_mask
].copy()

display(
    removed_lab_comment_rows["ResultLaboratoryCommentText"]
    .value_counts(dropna=False)
)

data_surface = data_surface.loc[
    ~exclude_mask
].copy()

print("Rows after:", len(data_surface))

Rows before: 95856
Rows to remove: 47


ResultLaboratoryCommentText
analyte detected in laboratory blank                                                            28
value reported not confirmable, interferencesee result laboratory comment                        6
interference from sample matrixsee result laboratory comment                                     4
result may be affected by interferencesee result laboratory comment                              3
result may be affected by interferencebelow the detection levelsee result laboratory comment     2
result may be affected by interferencevalue extrapolated at low endbelow the detection level     1
see result laboratory commentsample was dilutedinterference from sample matrix                   1
see result laboratory commentanalyte detected in laboratory blank                                1
holding time exceededanalyte detected in laboratory blank                                        1
Name: count, dtype: int64

Rows after: 95809


In [98]:
# I only use laboratory comments when they explicitly
# describe whether the analyte was detected or not detected.
#
# Detected ("Y"):
# - "below the reporting level but at or above the detection level"
# This means the analyte was found in the sample but the
# concentration was below the laboratory reporting threshold.
#
# Not detected ("N"):
# - "below the detection level"
# This means the analyte was not detected in the sample.
# All laboratory-comment rules are applied only to records where
# pfas_detected is still missing so that earlier classifications
# are never overwritten.

In [99]:
# I classify results as detected when the laboratory comment
# indicates the analyte was present above the detection level.

mask_lab_comment_detected = (
    data_surface["ResultLaboratoryCommentText"]
    .astype("string")
    .str.strip()
    .str.contains(
        "below the reporting level but at or above the detection level",
        na=False
    )
)

data_surface.loc[
    data_surface["pfas_detected"].isna()
    & mask_lab_comment_detected,
    "pfas_detected"
] = "Y"

In [100]:
print(
    "Rows classified as Y from laboratory comments:",
    (
        data_surface["pfas_detected"].eq("Y")
        & mask_lab_comment_detected
    ).sum()
)

Rows classified as Y from laboratory comments: 1950


In [101]:
# Laboratory comment not-detected rule


mask_lab_comment_detected = (
    data_surface["ResultLaboratoryCommentText"]
    .astype("string")
    .str.strip()
    .str.contains(
        "below the reporting level but at or above the detection level",
        case=False,
        na=False
    )
)

mask_lab_comment_not_detected = (
    data_surface["ResultLaboratoryCommentText"]
    .astype("string")
    .str.strip()
    .str.contains(
        "below the detection level",
        case=False,
        na=False
    )
    & ~mask_lab_comment_detected
)

data_surface.loc[
    data_surface["pfas_detected"].isna()
    & mask_lab_comment_not_detected,
    "pfas_detected"
] = "N"

In [102]:
print(
    "Rows classified as N from laboratory comments:",
    (
        data_surface["pfas_detected"].eq("N")
        & mask_lab_comment_not_detected
    ).sum()
)

Rows classified as N from laboratory comments: 814


### 3. Define rule based on ResultDetectionConditionText and ResultMeasureValue_text

In [103]:
# I review detection condition text when ResultMeasureValue is missing.

display(
    data_surface.loc[
        data_surface["ResultMeasureValue_text"].isna(),
        "ResultDetectionConditionText"
    ]
    .value_counts(dropna=False)
)

ResultDetectionConditionText
Not Detected                          30503
NaN                                   18462
Below Method Detection Limit          12197
Present Below Quantification Limit     1866
Not Detected at Reporting Limit         812
Below Reporting Limit                   289
Detected Not Quantified                   3
Name: count, dtype: int64

In [104]:
# I compare ResultMeasureValue_text against ResultDetectionConditionText.

pd.crosstab(
    data_surface["ResultDetectionConditionText"],
    data_surface["ResultMeasureValue_text"],
    dropna=False
)

ResultMeasureValue_text,##,0,0.00190000000000000,0.00200000000000000,0.00220000000000000,0.0023,0.00230000000000000,0.00250000000000000,0.00270000000000000,0.0028,...,< 2.1 ng/L,< 2.8 ng/L,< 2.9 ng/L,< 3.3 ng/L,< 4 ng/L,< 5 ng/L,<2.0,<MDL,ND,<NA>
ResultDetectionConditionText,,,,,,,,,,,,,,,,,,,,,
Below Method Detection Limit,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,12197
Below Reporting Limit,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,289
Detected Not Quantified,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,3
Not Detected,0,127,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,235,0,1022,30503
Not Detected at Reporting Limit,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,812
Present Above Quantification Limit,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,12,0,0,0
Present Below Quantification Limit,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,529,0,0,1866
NaN,1,0,1,2,2,1,1,2,1,1,...,2,2,2,2,2,1,0,2128,78,18462


In [105]:
# I use ResultDetectionConditionText and ResultMeasureValue_text only for
# rows that are still unclassified after the ResultCommentText and
# ResultLaboratoryCommentText rules.
#
# Detection assumptions:
#
# Detected ("Y"):
# - "Present Below Quantification Limit" with missing, ND, or less-than value
# - "Present Above Quantification Limit" with less-than value
# - "Detected Not Quantified" with missing or ND value
# - Any numeric ResultMeasureValue_num, if still unclassified
#
# Not detected ("N"):
# - "Below Method Detection Limit" with missing or ND value
# - "Below Reporting Limit" with missing or ND value
# - "Not Detected at Reporting Limit" with missing or ND value
# - "Not Detected" with missing, ND, or 0 value
# - "ND" with missing ResultDetectionConditionText
#
# Uncertain ("U"):
# - "Not Detected" with a less-than value, such as "<2.0"
# - less-than value with missing ResultDetectionConditionText
#
# I apply every rule only where pfas_detected is still missing so
# earlier classifications are not overwritten.

In [106]:
# I prepare masks for ResultDetectionConditionText and ResultMeasureValue_text.

condition_text = (
    data_surface["ResultDetectionConditionText"]
    .astype("string")
    .str.strip()
)

result_text = (
    data_surface["ResultMeasureValue_text"]
    .astype("string")
    .str.strip()
)

result_text_upper = result_text.str.upper()

is_result_missing = result_text.isna()

is_nd = result_text_upper.eq("ND")

is_zero_text = result_text.eq("0")

is_missing_or_nd = (
    is_result_missing
    | is_nd
)

is_missing_nd_or_zero = (
    is_result_missing
    | is_nd
    | is_zero_text
)

is_less_than = result_text.str.startswith("<", na=False)

is_numeric = data_surface["ResultMeasureValue_num"].notna()

In [107]:
# I prepare masks for ResultDetectionConditionText and ResultMeasureValue_text.

condition_text = (
    data_surface["ResultDetectionConditionText"]
    .astype("string")
    .str.strip()
)

result_text = (
    data_surface["ResultMeasureValue_text"]
    .astype("string")
    .str.strip()
)

result_text_upper = result_text.str.upper()

is_result_missing = result_text.isna()

is_nd = result_text_upper.eq("ND")

is_zero_text = result_text.eq("0")

is_missing_or_nd = (
    is_result_missing
    | is_nd
)

is_missing_nd_or_zero = (
    is_result_missing
    | is_nd
    | is_zero_text
)

is_less_than = result_text.str.startswith("<", na=False)

is_numeric = data_surface["ResultMeasureValue_num"].notna()

In [108]:
# Rule 3A
# Present Above Quantification Limit with a less-than value is still treated
# as detected because the detection condition says the analyte was present.

mask_above_quant_less_than_detected = (
    condition_text.eq("Present Above Quantification Limit")
    & is_less_than
)

mask_new_above_quant_less_than_detected = (
    data_surface["pfas_detected"].isna()
    & mask_above_quant_less_than_detected
)

print(
    "New Y from Present Above Quantification Limit with less-than value:",
    mask_new_above_quant_less_than_detected.sum()
)

data_surface.loc[
    mask_new_above_quant_less_than_detected,
    "pfas_detected"
] = "Y"

New Y from Present Above Quantification Limit with less-than value: 12


In [109]:
# Rule 3B
# Tony this and next one is NEW
# Clear not-detected conditions with missing or ND result values
# are treated as not detected.

mask_condition_not_detected = (
    condition_text.isin([
        "Below Method Detection Limit",
        "Not Detected at Reporting Limit",
        "Not Detected"
    ])
    & is_missing_or_nd
)

mask_new_condition_not_detected = (
    data_surface["pfas_detected"].isna()
    & mask_condition_not_detected
)

print(
    "New N from not-detected condition:",
    mask_new_condition_not_detected.sum()
)

data_surface.loc[
    mask_new_condition_not_detected,
    "pfas_detected"
] = "N"

New N from not-detected condition: 43654


In [110]:
# I treat results reported only as below the reporting limit
# as uncertain because this does not clearly establish whether
# the analyte was detected.

mask_below_reporting_limit = (
    condition_text.eq("Below Reporting Limit")
    & is_missing_or_nd
)

mask_new_below_reporting_limit = (
    data_surface["pfas_detected"].isna()
    & mask_below_reporting_limit
)

print(
    "New U from below reporting limit:",
    mask_new_below_reporting_limit.sum()
)

data_surface.loc[
    mask_new_below_reporting_limit,
    "pfas_detected"
] = "U"

New U from below reporting limit: 289


In [111]:
# Rule 3C
# Not Detected with result value 0 is treated as not detected.

mask_not_detected_zero = (
    condition_text.eq("Not Detected")
    & is_zero_text
)

mask_new_not_detected_zero = (
    data_surface["pfas_detected"].isna()
    & mask_not_detected_zero
)

print("New N from Not Detected with 0 result:", mask_new_not_detected_zero.sum())

data_surface.loc[
    mask_new_not_detected_zero,
    "pfas_detected"
] = "N"

New N from Not Detected with 0 result: 127


In [112]:
# Rule 3D
# Not Detected with a less-than value is uncertain.

mask_not_detected_less_than = (
    condition_text.eq("Not Detected")
    & is_less_than
)

mask_new_not_detected_less_than = (
    data_surface["pfas_detected"].isna()
    & mask_not_detected_less_than
)

print("New U from Not Detected with less-than value:", mask_new_not_detected_less_than.sum())

data_surface.loc[
    mask_new_not_detected_less_than,
    "pfas_detected"
] = "U"

New U from Not Detected with less-than value: 235


In [113]:
# Rule 3E
# Any remaining numeric result value is treated as detected.

mask_numeric_detected = is_numeric

mask_new_numeric_detected = (
    data_surface["pfas_detected"].isna()
    & mask_numeric_detected
)

print("New Y from numeric result value:", mask_new_numeric_detected.sum())

data_surface.loc[
    mask_new_numeric_detected,
    "pfas_detected"
] = "Y"

New Y from numeric result value: 0


In [114]:
# Rule 3F
# ND with missing ResultDetectionConditionText is treated as not detected.

mask_nd_missing_condition = (
    is_nd
    & condition_text.isna()
)

mask_new_nd_missing_condition = (
    data_surface["pfas_detected"].isna()
    & mask_nd_missing_condition
)

print("New N from ND with missing detection condition:", mask_new_nd_missing_condition.sum())

data_surface.loc[
    mask_new_nd_missing_condition,
    "pfas_detected"
] = "N"

New N from ND with missing detection condition: 78


In [115]:
# Rule 3H
# A less-than value with missing ResultDetectionConditionText is uncertain.

mask_less_than_missing_condition = (
    is_less_than
    & condition_text.isna()
)

mask_new_less_than_missing_condition = (
    data_surface["pfas_detected"].isna()
    & mask_less_than_missing_condition
)

print("New U from less-than value with missing condition:", mask_new_less_than_missing_condition.sum())

data_surface.loc[
    mask_new_less_than_missing_condition,
    "pfas_detected"
] = "U"

New U from less-than value with missing condition: 0


In [116]:
# Flagging

In [117]:
# Rule 3 I
#Tony this is NEW
# I review the source and qualifier values for records
# that would be classified as not detected under Rule 3I.

qualifier_clean = (
    data_surface["MeasureQualifierCode"]
    .astype("string")
    .str.upper()
    .str.strip()
)

mask_u_qualifier = qualifier_clean.str.contains("U", na=False)

display(
    data_surface.loc[
        mask_u_qualifier,
        [
            "source",
            "MeasureQualifierCode",
            "ResultDetectionConditionText",
            "ResultMeasureValue_text",
            "ResultMeasureValue_num"
        ]
    ]
    .groupby(
        ["source", "MeasureQualifierCode"],
        dropna=False
    )
    .size()
    .reset_index(name="records")
    .sort_values("records", ascending=False)
)

,source,MeasureQualifierCode,records
6,WQP,U,19883
0,Maine DEP,U,11285
2,Massachusetts,U,3594
1,Maine DEP,UJ,3433
5,WQP,J;U,246
4,Massachusetts,U cn,10
3,Massachusetts,U *+,4


In [118]:
# I review which organizations use the U qualifier in the WQP data.

display(
    data_surface.loc[
        data_surface["source"].eq("WQP")
        & qualifier_clean.eq("U"),
        "OrganizationFormalName"
    ]
    .value_counts(dropna=False)
    .reset_index(name="records")
)

,OrganizationFormalName,records
0,Minnesota Pollution Control Agency - Ambient S...,11889
1,New York State Dec Division Of Water,4802
2,Delaware River Basin Commission,2932
3,NM Environmental Dept./SWQB,260


In [119]:
# For Minnesota Pollution Control Agency : found this:https://www.pca.state.mn.us/sites/default/files/tdr-g1-18.pdf, Appendix B
# Rule 3I-A - Minnesota Pollution Control Agency

# For Minnesota Pollution Control Agency records,
# the qualifier U means the analyte was not detected
# at the reporting limit, so I classify these records as N.

qualifier_clean = (
    data_surface["MeasureQualifierCode"]
    .astype("string")
    .str.upper()
    .str.strip()
)

mask_minnesota_u = (
    data_surface["OrganizationFormalName"]
    .astype("string")
    .str.contains(
        "Minnesota Pollution Control Agency",
        case=False,
        na=False
    )
    & qualifier_clean.eq("U")
)

mask_new_minnesota_u = (
    data_surface["pfas_detected"].isna()
    & mask_minnesota_u
)

print(
    "Minnesota rows assigned N from U qualifier:",
    mask_new_minnesota_u.sum()
)

data_surface.loc[
    mask_new_minnesota_u,
    "pfas_detected"
] = "N"

Minnesota rows assigned N from U qualifier: 0


In [120]:
# Rule 3I-B - New York State DEC Division of Water : found this:https://dec.ny.gov/sites/default/files/2024-05/eddmanualnewv2024.pdf, page 58

# For New York State DEC Division of Water records,
# the qualifier U means the analyte was analyzed for but not detected,
# so I classify these records as N.

mask_new_york_u = (
    data_surface["OrganizationFormalName"]
    .astype("string")
    .str.strip()
    .eq("New York State Dec Division Of Water")
    & qualifier_clean.eq("U")
)

mask_new_new_york_u = (
    data_surface["pfas_detected"].isna()
    & mask_new_york_u
)

print(
    "New York DEC rows assigned N from U qualifier:",
    mask_new_new_york_u.sum()
)

data_surface.loc[
    mask_new_new_york_u,
    "pfas_detected"
] = "N"

New York DEC rows assigned N from U qualifier: 0


In [121]:
# for DRBC using this source:https://nj.gov/drbc/programs/quality/1-4Dioxane.html
# Rule 3I-C - Delaware River Basin Commission

# For Delaware River Basin Commission records, the qualifier U
# means the analyte was analyzed for but not detected.
# I therefore classify these records as N.

qualifier_clean = (
    data_surface["MeasureQualifierCode"]
    .astype("string")
    .str.upper()
    .str.strip()
)

mask_drbc_u = (
    data_surface["OrganizationFormalName"]
    .astype("string")
    .str.strip()
    .eq("Delaware River Basin Commission")
    & qualifier_clean.eq("U")
)

mask_new_drbc_u = (
    data_surface["pfas_detected"].isna()
    & mask_drbc_u
)

print(
    "Delaware River Basin Commission rows assigned N from U qualifier:",
    mask_new_drbc_u.sum()
)

data_surface.loc[
    mask_new_drbc_u,
    "pfas_detected"
] = "N"

Delaware River Basin Commission rows assigned N from U qualifier: 0


In [122]:
# Rule 3I- D - NM source:https://www.env.nm.gov/wp-content/uploads/sites/18/2021/11/EPA-Signed-2021-09-22-SWQB-QAPP-Final-signed_SWQB.pdf page 48 of 82

# For NM Environmental Dept./SWQB records, the qualifier U
# means the analyte was not detected. I therefore classify
# these records as N.

qualifier_clean = (
    data_surface["MeasureQualifierCode"]
    .astype("string")
    .str.upper()
    .str.strip()
)

mask_nm_u = (
    data_surface["OrganizationFormalName"]
    .astype("string")
    .str.strip()
    .eq("NM Environmental Dept./SWQB")
    & qualifier_clean.eq("U")
)

mask_new_nm_u = (
    data_surface["pfas_detected"].isna()
    & mask_nm_u
)

print(
    "NM Environmental Dept./SWQB rows assigned N from U qualifier:",
    mask_new_nm_u.sum()
)

data_surface.loc[
    mask_new_nm_u,
    "pfas_detected"
] = "N"

NM Environmental Dept./SWQB rows assigned N from U qualifier: 260


In [123]:
# Rule 3I- F- Massachusetts

# For Massachusetts PFAS records, the laboratory qualifier U
# means not detected. I therefore classify these records as N.

qualifier_clean = (
    data_surface["MeasureQualifierCode"]
    .astype("string")
    .str.upper()
    .str.strip()
)

mask_massachusetts_u = (
    data_surface["source"]
    .astype("string")
    .str.contains(
        "Massachusetts",
        case=False,
        na=False
    )
    & qualifier_clean.eq("U")
)

mask_new_massachusetts_u = (
    data_surface["pfas_detected"].isna()
    & mask_massachusetts_u
)

print(
    "Massachusetts rows assigned N from U qualifier:",
    mask_new_massachusetts_u.sum()
)

data_surface.loc[
    mask_new_massachusetts_u,
    "pfas_detected"
] = "N"

Massachusetts rows assigned N from U qualifier: 0


In [124]:
# I review all remaining records with an exact U qualifier that
# have not yet been assigned a PFAS detection status.

qualifier_clean = (
    data_surface["MeasureQualifierCode"]
    .astype("string")
    .str.upper()
    .str.strip()
)

remaining_u = data_surface.loc[
    data_surface["pfas_detected"].isna()
    & qualifier_clean.eq("U")
].copy()

print(
    "Remaining unclassified records with U qualifier:",
    len(remaining_u)
)

# I summarize the remaining U-qualified records by organization
# so I can determine how each organization defines the qualifier.

remaining_u_by_org = (
    remaining_u
    .groupby(
        [
            "OrganizationIdentifier",
            "OrganizationFormalName"
        ],
        dropna=False
    )
    .size()
    .reset_index(name="records")
    .sort_values(
        "records",
        ascending=False
    )
)

display(remaining_u_by_org)

Remaining unclassified records with U qualifier: 11285


,OrganizationIdentifier,OrganizationFormalName,records
0,NaN,Maine Department of Environmental Protection,11285


In [125]:
# I confirm that no records with an exact U qualifier remain
# unclassified after applying the organization-specific rules.

qualifier_clean = (
    data_surface["MeasureQualifierCode"]
    .astype("string")
    .str.upper()
    .str.strip()
)

mask_remaining_u = (
    data_surface["pfas_detected"].isna()
    & qualifier_clean.eq("U")
)

print(
    "Remaining unclassified records with U qualifier:",
    mask_remaining_u.sum()
)

Remaining unclassified records with U qualifier: 11285


In [126]:
# Tony this is NEW
#Rule 4 
# I classify remaining records with a U qualifier as not detected.
# These records generally have no reported result value but do have
# a reporting limit or detection limit.

qualifier_clean = (
    data_surface["MeasureQualifierCode"]
    .astype("string")
    .str.upper()
    .str.strip()
)

mask_u_qualifier = qualifier_clean.str.contains("U", na=False)

data_surface.loc[
    data_surface["pfas_detected"].isna()
    & mask_u_qualifier,
    "pfas_detected"
] = "N"

print(
    "Rows assigned N from U qualifier:",
    (
        data_surface["pfas_detected"].eq("N")
        & mask_u_qualifier
    ).sum()
)

display(data_surface["pfas_detected"].value_counts(dropna=False))

Rows assigned N from U qualifier: 38166


pfas_detected
N      65769
Y      16519
U      11628
NaN     1893
Name: count, dtype: int64

In [127]:
# I identify the remaining unclassified records with a U qualifier
# before assigning their detection status.

mask_new_u_qualifier = (
    data_surface["pfas_detected"].isna()
    & qualifier_clean.str.contains("U", na=False)
)

print(
    "New rows assigned N from U qualifier:",
    mask_new_u_qualifier.sum()
)

# I classify these remaining U-qualified records as not detected.

data_surface.loc[
    mask_new_u_qualifier,
    "pfas_detected"
] = "N"

New rows assigned N from U qualifier: 0


In [128]:
# I check whether my general PFAS detection rules correctly
# classified all Maine DEP records.

mask_maine = (
    data_surface["source"].eq("Maine DEP")
)

display(
    data_surface.loc[
        mask_maine,
        "pfas_detected"
    ]
    .value_counts(dropna=False)
)

# I cross-check Maine's final detection status against
# the validation qualifiers.

display(
    pd.crosstab(
        data_surface.loc[
            mask_maine,
            "MeasureQualifierCode"
        ].fillna("MISSING"),
        data_surface.loc[
            mask_maine,
            "pfas_detected"
        ].fillna("UNCLASSIFIED"),
        margins=True
    )
)

pfas_detected
N    14718
Y     5508
Name: count, dtype: int64

pfas_detected,N,Y,All
MeasureQualifierCode,,,
E,0,1,1
J,0,2971,2971
MISSING,0,2536,2536
U,11285,0,11285
UJ,3433,0,3433
All,14718,5508,20226


In [129]:
# I check whether the remaining unclassified records truly contain
# no PFAS result information before removing them.

remaining_unclassified = data_surface.loc[
    data_surface["pfas_detected"].isna()
].copy()

result_information_columns = [
    "ResultMeasureValue_text",
    "ResultMeasureValue_num",
    "MeasureQualifierCode",
    "ResultDetectionConditionText",
    "ResultLaboratoryCommentText",
    "ResultCommentText",
    "DetectionQuantitationLimitMeasure/MeasureValue"
]

print("Remaining unclassified rows:", len(remaining_unclassified))

for col in result_information_columns:
    print(
        col,
        "non-missing:",
        remaining_unclassified[col].notna().sum()
    )

Remaining unclassified rows: 1893
ResultMeasureValue_text non-missing: 529
ResultMeasureValue_num non-missing: 0
MeasureQualifierCode non-missing: 382
ResultDetectionConditionText non-missing: 1892
ResultLaboratoryCommentText non-missing: 0
ResultCommentText non-missing: 356
DetectionQuantitationLimitMeasure/MeasureValue non-missing: 1892


In [130]:
# I identify unclassified records that still contain
# some potentially useful PFAS result information.

has_result_information = (
    remaining_unclassified[
        result_information_columns
    ]
    .notna()
    .any(axis=1)
)

print(
    "Unclassified rows with some result information:",
    has_result_information.sum()
)

Unclassified rows with some result information: 1892


In [131]:
# I summarize the remaining unclassified records by
# detection condition and qualifier before deciding how to classify them.

remaining_unclassified = data_surface.loc[
    data_surface["pfas_detected"].isna()
].copy()

display(
    remaining_unclassified.groupby(
        [
            "ResultDetectionConditionText",
            "MeasureQualifierCode"
        ],
        dropna=False
    )
    .size()
    .reset_index(name="records")
    .sort_values("records", ascending=False)
)

,ResultDetectionConditionText,MeasureQualifierCode,records
5,Present Below Quantification Limit,NaN,1507
4,Present Below Quantification Limit,J,365
1,Present Below Quantification Limit,B;J,8
2,Present Below Quantification Limit,EMPC;J,8
0,Detected Not Quantified,NaN,3
3,Present Below Quantification Limit,H;J,1
6,NaN,NaN,1


In [132]:
# I review the most common result comments among the
# remaining unclassified records.

display(
    remaining_unclassified["ResultCommentText"]
    .value_counts(dropna=False)
    .reset_index(name="records")
    .head(30)
)

,ResultCommentText,records
0,NaN,1537
1,CAS#: 108427-53-8,30
2,CAS#: 72007-68-2,29
3,CAS#: 45187-15-3,27
4,CAS#: 45167-47-3,26
5,CAS#: 45048-62-2,25
6,CAS#: 120885-29-2,23
7,CAS# 45048-62-2,19
8,CAS# 108427-53-8,18
9,CAS#: 73829-36-4,13


In [133]:
# I summarize the remaining unclassified records by
# detection condition and qualifier before deciding how to classify them.

remaining_unclassified = data_surface.loc[
    data_surface["pfas_detected"].isna()
].copy()

display(
    remaining_unclassified.groupby(
        [
            "ResultDetectionConditionText",
            "MeasureQualifierCode"
        ],
        dropna=False
    )
    .size()
    .reset_index(name="records")
    .sort_values("records", ascending=False)
)

,ResultDetectionConditionText,MeasureQualifierCode,records
5,Present Below Quantification Limit,NaN,1507
4,Present Below Quantification Limit,J,365
1,Present Below Quantification Limit,B;J,8
2,Present Below Quantification Limit,EMPC;J,8
0,Detected Not Quantified,NaN,3
3,Present Below Quantification Limit,H;J,1
6,NaN,NaN,1


In [134]:
# Rule 4 - Present Below Quantification Limit

# I classify remaining records as detected when
# ResultDetectionConditionText explicitly says that the analyte
# was present below the quantification limit.
#
# I exclude records with a U qualifier because those records
# are reviewed separately.

qualifier_clean = (
    data_surface["MeasureQualifierCode"]
    .astype("string")
    .str.upper()
    .str.strip()
)

mask_present_below_quantification = (
    data_surface["ResultDetectionConditionText"]
    .astype("string")
    .str.strip()
    .eq("Present Below Quantification Limit")
)

mask_new_present_below_quantification = (
    data_surface["pfas_detected"].isna()
    & mask_present_below_quantification
    & ~qualifier_clean.str.contains("U", na=False)
)

print(
    "New rows assigned Y from Present Below Quantification Limit:",
    mask_new_present_below_quantification.sum()
)

data_surface.loc[
    mask_new_present_below_quantification,
    "pfas_detected"
] = "Y"

New rows assigned Y from Present Below Quantification Limit: 1889


In [135]:
# I check whether any Present Below Quantification Limit records
# remain unclassified after applying Rule 4.

remaining_present_below = data_surface.loc[
    data_surface["pfas_detected"].isna()
    & data_surface["ResultDetectionConditionText"]
        .astype("string")
        .str.strip()
        .eq("Present Below Quantification Limit")
].copy()

print(
    "Remaining unclassified Present Below Quantification Limit records:",
    len(remaining_present_below)
)

display(
    remaining_present_below[
        [
            "ResultMeasureValue",
            "ResultMeasureValue_text",
            "ResultMeasureValue_num",
            "MeasureQualifierCode",
            "ResultDetectionConditionText"
        ]
    ]
)

Remaining unclassified Present Below Quantification Limit records: 0


,ResultMeasureValue,ResultMeasureValue_text,ResultMeasureValue_num,MeasureQualifierCode,ResultDetectionConditionText


In [136]:
# I review the final detection counts after ResultDetectionConditionText
# and ResultMeasureValue_text rules.

display(data_surface["pfas_detected"].value_counts(dropna=False))

pfas_detected
N      65769
Y      18408
U      11628
NaN        4
Name: count, dtype: int64

In [137]:
# I inspect the result-related fields for the remaining
# unclassified records.

result_information_columns = [
    "ResultMeasureValue_text",
    "ResultMeasureValue_num",
    "MeasureQualifierCode",
    "ResultDetectionConditionText",
    "ResultLaboratoryCommentText",
    "ResultCommentText",
    "DetectionQuantitationLimitTypeName",
    "DetectionQuantitationLimitMeasure/MeasureValue",
    "DetectionQuantitationLimitMeasure/MeasureUnitCode"
]

remaining_unclassified = data_surface.loc[
    data_surface["pfas_detected"].isna(),
    result_information_columns
].copy()

print(
    "Remaining unclassified rows:",
    len(remaining_unclassified)
)

display(
    remaining_unclassified.head(10)
)

Remaining unclassified rows: 4


,ResultMeasureValue_text,ResultMeasureValue_num,MeasureQualifierCode,ResultDetectionConditionText,ResultLaboratoryCommentText,ResultCommentText,DetectionQuantitationLimitTypeName,DetectionQuantitationLimitMeasure/MeasureValue,DetectionQuantitationLimitMeasure/MeasureUnitCode
522,<NA>,<NA>,NaN,Detected Not Quantified,NaN,NaN,Method Detection Limit (MDL),0.183,ng/l
1111,<NA>,<NA>,NaN,Detected Not Quantified,NaN,NaN,Method Detection Limit (MDL),0.205,ng/l
8920,<NA>,<NA>,NaN,Detected Not Quantified,NaN,NaN,Method Detection Limit (MDL),0.184,ng/l
94850,<NA>,<NA>,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [138]:
# I check the text result values for the remaining records reported as
# Present Below Quantification Limit.

check_present_below = data_surface.loc[
    data_surface["pfas_detected"].isna()
    & data_surface["ResultDetectionConditionText"]
        .astype("string")
        .str.strip()
        .eq("Present Below Quantification Limit")
].copy()

print(
    "Remaining Present Below Quantification Limit records:",
    len(check_present_below)
)

print("\nResultMeasureValue_text:")
display(
    check_present_below["ResultMeasureValue_text"]
    .value_counts(dropna=False)
)

print("\nResultMeasureValue_num:")
display(
    check_present_below["ResultMeasureValue_num"]
    .value_counts(dropna=False)
)

print("\nMeasureQualifierCode:")
display(
    check_present_below["MeasureQualifierCode"]
    .value_counts(dropna=False)
)

Remaining Present Below Quantification Limit records: 0

ResultMeasureValue_text:


Series([], Name: count, dtype: Int64)


ResultMeasureValue_num:


Series([], Name: count, dtype: Int64)


MeasureQualifierCode:


Series([], Name: count, dtype: int64)

In [139]:
# Rule 3J
# I classify records as detected when the detection condition
# explicitly states that the PFAS analyte was detected or present,
# even when a quantified concentration was not reported.

detection_condition_clean = (
    data_surface["ResultDetectionConditionText"]
    .astype("string")
    .str.strip()
)

detected_not_quantified_conditions = [
    "Detected Not Quantified",
    "Present Below Quantification Limit"
]

mask_detected_not_quantified = (
    data_surface["pfas_detected"].isna()
    & detection_condition_clean.isin(
        detected_not_quantified_conditions
    )
)

print(
    "Rows assigned Y from detected/present but not quantified:",
    mask_detected_not_quantified.sum()
)

data_surface.loc[
    mask_detected_not_quantified,
    "pfas_detected"
] = "Y"

display(
    data_surface["pfas_detected"]
    .value_counts(dropna=False)
)

Rows assigned Y from detected/present but not quantified: 3


pfas_detected
N      65769
Y      18411
U      11628
NaN        1
Name: count, dtype: int64

In [140]:
remaining_unclassified = data_surface.loc[
    data_surface["pfas_detected"].isna()
].copy()

print(
    "Remaining unclassified rows:",
    len(remaining_unclassified)
)

Remaining unclassified rows: 1


In [141]:
# I remove the final record that could not be classified
# as either detected or not detected.

rows_before = len(data_surface)

data_surface = data_surface.loc[
    data_surface["pfas_detected"].notna()
].copy()

print("Rows removed:", rows_before - len(data_surface))
print("Rows remaining:", len(data_surface))

Rows removed: 1
Rows remaining: 95808


In [142]:
# I check whether the previously unclassified record
# is still present in the dataset.

if 94850 in data_surface.index:
    print("Row 94850 is still in data_surface.")
    
    display(
        data_surface.loc[
            [94850],
            [
                "MonitoringLocationIdentifier",
                "ActivityStartDate",
                "CharacteristicName",
                "ResultMeasureValue",
                "ResultDetectionConditionText",
                "MeasureQualifierCode",
                "pfas_detected"
            ]
        ]
    )
else:
    print("Row 94850 is no longer in data_surface.")

Row 94850 is no longer in data_surface.


In [143]:
# I confirm the PFAS detection classifications for
# Waterkeeper Phase II records.

display(
    data_surface.loc[
        data_surface["source"].eq("Waterkeeper Phase II"),
        "pfas_detected"
    ]
    .value_counts(dropna=False)
)

pfas_detected
N    2264
Y     616
Name: count, dtype: int64

In [144]:
# I temporarily classify the remaining unassigned records as uncertain.
# These records do not contain enough information to determine whether
# PFAS was detected or not detected.

data_surface.loc[
    data_surface["pfas_detected"].isna(),
    "pfas_detected"
] = "U"

print(
    "Rows assigned U:",
    data_surface["pfas_detected"].eq("U").sum()
)

display(
    data_surface["pfas_detected"]
    .value_counts(dropna=False)
)

Rows assigned U: 11628


pfas_detected
N    65769
Y    18411
U    11628
Name: count, dtype: int64

In [145]:
print(phase1_long.shape)
print(phase2_long.shape)
print(ma_long.shape)

print(data_surface.shape)

(12312, 14)
(2880, 14)
(4240, 24)
(95808, 144)


## 6.Standardize columns

In [146]:
data_surface["ActivityLocation/LatitudeMeasure"] = pd.to_numeric(
    data_surface["ActivityLocation/LatitudeMeasure"],
    errors="coerce"
)

data_surface["ActivityLocation/LongitudeMeasure"] = pd.to_numeric(
    data_surface["ActivityLocation/LongitudeMeasure"],
    errors="coerce"
)

data_surface["ResultMeasureValue_num"] = pd.to_numeric(
    data_surface["ResultMeasureValue_text"],
    errors="coerce"
)

data_surface["pfas_detected"] = (
    data_surface["pfas_detected"]
    .astype("string")
    .str.strip()
    .str.upper()
)

print(data_surface.shape)
display(data_surface["source"].value_counts(dropna=False))
display(data_surface["pfas_detected"].value_counts(dropna=False))

(95808, 144)


source
WQP                     56151
Maine DEP               20226
Waterkeeper Phase I     12311
Massachusetts            4240
Waterkeeper Phase II     2880
Name: count, dtype: int64

pfas_detected
N    65769
Y    18411
U    11628
Name: count, dtype: Int64

## 7. Standrdizing results based on DetectionQuantitationLimitMeasure/MeasureUnitCode

In [147]:
display(
    data_surface["StatisticalBaseCode"]
    .value_counts(dropna=False)
)

StatisticalBaseCode
NaN    95808
Name: count, dtype: int64

In [148]:
display(
    data_surface["DetectionQuantitationLimitMeasure/MeasureUnitCode"]
    .value_counts(dropna=False)
)

DetectionQuantitationLimitMeasure/MeasureUnitCode
ng/L    48425
ng/l    23243
NaN     22637
ug/L     1023
ng        480
Name: count, dtype: int64

In [149]:
# I remove rows reported as percent recovery because these are
# laboratory QA/QC performance measurements rather than PFAS
# concentrations in environmental samples.

recovery_mask = (
    data_surface["ResultMeasure/MeasureUnitCode"]
    == "% recovery"
)

print("Rows before:", len(data_surface))
print("Rows removed:", recovery_mask.sum())

data_surface = data_surface.loc[
    ~recovery_mask
].copy()

print("Rows after:", len(data_surface))

Rows before: 95808
Rows removed: 288
Rows after: 95520


In [150]:
# Tony: this is new

In [151]:
display(
    data_surface["ResultMeasure/MeasureUnitCode"]
    .value_counts(dropna=False)
)

display(
    data_surface["DetectionQuantitationLimitMeasure/MeasureUnitCode"]
    .value_counts(dropna=False)
)

ResultMeasure/MeasureUnitCode
NaN     62180
ng/L    28930
ng/l     4209
ug/L      148
ng         53
Name: count, dtype: int64

DetectionQuantitationLimitMeasure/MeasureUnitCode
ng/L    48425
ng/l    23243
NaN     22349
ug/L     1023
ng        480
Name: count, dtype: int64

In [152]:
# I convert PFAS result values from ug/L to ng/L.
# 1 ug/L = 1,000 ng/L.

ug_result_mask = (
    data_surface["ResultMeasure/MeasureUnitCode"]
    .eq("ug/L")
    &
    data_surface["ResultMeasureValue"].notna()
)

print("Rows being converted:", ug_result_mask.sum())

data_surface.loc[
    ug_result_mask,
    "ResultMeasureValue"
] = (
    pd.to_numeric(
        data_surface.loc[
            ug_result_mask,
            "ResultMeasureValue"
        ],
        errors="coerce"
    ) * 1000
)

data_surface.loc[
    ug_result_mask,
    "ResultMeasure/MeasureUnitCode"
] = "ng/L"

print(
    data_surface["ResultMeasure/MeasureUnitCode"]
    .value_counts(dropna=False)
)

Rows being converted: 147
ResultMeasure/MeasureUnitCode
NaN     62180
ng/L    29077
ng/l     4209
ng         53
ug/L        1
Name: count, dtype: int64


In [153]:
display(
    data_surface.loc[
        data_surface["ResultMeasure/MeasureUnitCode"] == "ug/L",
        [
            "CharacteristicName",
            "ResultMeasureValue",
            "ResultMeasure/MeasureUnitCode",
            "DetectionQuantitationLimitMeasure/MeasureValue",
            "DetectionQuantitationLimitMeasure/MeasureUnitCode",
            "pfas_detected"
        ]
    ]
)

,CharacteristicName,ResultMeasureValue,ResultMeasure/MeasureUnitCode,DetectionQuantitationLimitMeasure/MeasureValue,DetectionQuantitationLimitMeasure/MeasureUnitCode,pfas_detected
18883,Perfluoropentanesulfonic acid,NaN,ug/L,0.69,ug/L,N


In [154]:
# I think ng is typo and I did fix ug/l

In [155]:
# FINAL PFAS DETECTION CLEANUP

# I classify all remaining uncertain PFAS detection results as
# not detected before saving the final cleaned dataset.

mask_final_u = (
    data_surface["pfas_detected"]
    .astype("string")
    .str.upper()
    .str.strip()
    .eq("U")
)

print(
    "Records changed from U to N:",
    mask_final_u.sum()
)

data_surface.loc[
    mask_final_u,
    "pfas_detected"
] = "N"

# I confirm the final PFAS detection classifications.

display(
    data_surface["pfas_detected"]
    .value_counts(dropna=False)
)

Records changed from U to N: 11628


pfas_detected
N    77397
Y    18123
Name: count, dtype: Int64

## 7. Save final cleaned analysis file

In [156]:
output_path = file_path + "saved files/Notebook2/pfas_cleaned_analysis_file.csv"

data_surface.to_csv(output_path, index=False)

print("Saved:", output_path)
print("Rows:", len(data_surface))

Saved: G:/My Drive/Projects/Large projects/PFAS in water/saved files/Notebook2/pfas_cleaned_analysis_file.csv
Rows: 95520
